# Faster R-CNN · 3차 클래스 불균형 실험

**v3 × baseline e10 기준 · rfs 9 / weight 10 / both 9 epoch · seed 0**

2차 참고 노트북의 데이터·모델·학습·평가 계약을 유지하고, 팀 `balance.py`의 RFS와 RoI 클래스 가중치를 연결합니다. under200, 장기 학습, baseline 재학습은 포함하지 않습니다.

| 조건 | 학습 목록 | RoI 분류 가중치 | epoch |
| --- | --- | --- | ---: |
| rfs | seed 0의 고정 RFS 목록 | 없음 | 9 |
| weight | 원래 v3 train | power 0.5 | 10 |
| both | rfs와 같은 목록 | 샘플링 후 객체 수, power 0.5 | 9 |

**환경은 사용자가 지정한 현재 uv.lock을 따릅니다.** 2차 실제 환경(torch 2.11.0 / torchvision 0.26.0)과 현재 잠금 환경(작성 시 2.14.1 / 0.29.1)은 다릅니다. baseline 대비 변화에는 환경 차이도 섞이므로 불균형 기법만의 효과로 단정하지 않습니다. baseline best를 새 실험의 초기값으로 쓰지 않습니다.

power 0.5는 팀의 balance.py 사용 예·3차 스모크 노트북 CLS_PW에서 온 값이며, 2026-11-04 사용자가 채택했습니다. 최적값 탐색 결과가 아닙니다. 공용 가중치 식에는 별도 상한이 없습니다.

**실행 순서:** 1 설정 → 2 저장소·잠금 환경 준비/확인 → 3 함수 정의 → 4 Drive 연결 → 5 데이터 준비 → 6 baseline·목록 확인 → 7 선택 검사 → 8 본 학습 또는 9 재개 → 10 best 평가·비교 → 11 선택적 test 추론·CSV 생성.

학습·추론·설치·데이터 전개·CSV 생성 스위치는 기본 False입니다. 실행 중인 다른 학습 커널에서 사용하지 말고 별도 Colab GPU 세션을 사용합니다. 이 노트북은 커널을 자동 재시작하지 않습니다. git 브랜치 변경·강제 초기화·기존 파일 삭제·Kaggle 제출도 자동 실행하지 않습니다.

## 1. 실행 경로와 확정 조건
Colab에 연결되는 공유 폴더 바로가기 경로를 확인하세요. 로컬에서는 열람·수치 검사만 하며 학습 준비는 Colab에서 수행합니다.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import shutil
import tempfile
import tomllib

PLATFORM = "colab" if "google.colab" in sys.modules or Path("/content").exists() else "local"
REPO_ROOT = Path("/content/pill-object-detection" if PLATFORM == "colab" else
                 "/Users/codeit/Desktop/codeit/Project1/pill-object-detection")
REPOSITORY_URL = "https://github.com/codeit-ai-15th/pill-object-detection.git"
BRANCH = "exp/faster-rcnn-exp3"
DRIVE_BASE = Path("/content/drive/MyDrive/코드잇 ;)/Project1_pill-object-detection/pill data" if PLATFORM == "colab" else
    "/Users/codeit/Library/CloudStorage/GoogleDrive-khsbear0115@gmail.com/My Drive/코드잇 ;)/Project1_pill-object-detection/pill data")
ZIP_ROOT = DRIVE_BASE / "data"
BASELINE_DIR = DRIVE_BASE / "rw_runs/v3_frcnn_e10_s0"
RUN_ROOT = DRIVE_BASE / "cj_runs/exp3"
DATASET_ROOT = Path("/content/exp3_datasets" if PLATFORM == "colab" else REPO_ROOT / "data/processed/exp3")
CONDITIONS = ("rfs", "weight", "both")
WEIGHT_POWER = 0.5
RFS_THRESHOLD = None  # 팀 기본값: 모델 클래스 이미지 빈도의 중앙값. 준비 시 실제 숫자를 기록한다.
RFS_SEED = 0
RUN_SUFFIX = ""  # 새 독립 실험이 필요하면 구분자를 지정한다. 기존 결과를 지우지 않는다.
MONITOR_CLASSES = (2483, 19552, 12778, 18147)
BASELINE_PUBLIC_SCORE = 0.59750  # 제공된 실제 제출 기록. scores.json의 kaggle와 구분.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
print("현재 폴더:", REPO_ROOT, "\n결과:", RUN_ROOT, "\nbaseline:", BASELINE_DIR)

## 2. 현재 uv.lock 환경 준비와 검증
노트북을 사용자가 커밋·push한 뒤 새 Colab에서 준비합니다. uv가 없다면 저장소 README의 uv 설치를 먼저 직접 수행하세요. 설치 셀은 프로젝트 파일과 lock을 수정하지 않고 lock을 내보내 현재 Python에 적용합니다. 설치 전에 torch를 import하지 마세요. 현재 커널에 이미 torch가 로드됐다면 새 세션에서 준비합니다. 설치 후 커널 재시작이 필요한 경우 사용자가 직접 판단하며, 실행 중인 학습 커널은 건드리지 않습니다.

CUDA PyTorch의 배포 출처와 wheel 해시를 유지하도록 현재 uv.lock을 `pylock.toml`로 내보내 설치합니다. `requirements.txt` 변환은 사용하지 않습니다. 환경 준비 파일과 uv 출력은 표시된 `/tmp/frcnn_exp3_setup_.../` 폴더에 보존되며, 실패하면 마지막 오류와 로그 파일 경로를 함께 출력합니다. 실패한 설치의 원인은 uv 오류 원문으로 확인합니다.


**설치가 성공하면 현재 3차 준비용 커널만 직접 Restart합니다.** 서버 삭제·VM 초기화는 하지 않습니다. 재시작 후 1번 설정을 실행하고, 설치 스위치 두 개는 False로 둔 채 2번 환경 확인 셀을 실행합니다. 설치를 반복하면 다시 재시작이 필요합니다. 다른 학습이 진행 중인 커널에는 이 준비 과정을 적용하지 않습니다.


In [ ]:
from collections import deque


def run_setup_command(command, stage, log_dir):
    """uv 출력을 화면과 파일에 함께 남기고 실패하면 실제 오류와 로그 경로를 안내한다."""
    log_path = Path(log_dir) / f"{stage}.log"
    tail = deque(maxlen=60)
    with log_path.open("w", encoding="utf-8") as log:
        with subprocess.Popen(command, cwd=REPO_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding="utf-8", errors="replace", bufsize=1) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
                tail.append(line)
            returncode = process.wait()
    if returncode:
        details = "".join(tail).strip()
        raise RuntimeError(
            f"{stage} 실패(exit {returncode}). 아래 uv 오류와 로그를 확인하세요.\n"
            f"로그: {log_path}\n{details}\n"
            "설정·lock을 임의로 바꾸거나 학습으로 넘어가지 말고 오류 부분을 확인하세요."
        )
    return log_path


PREPARE_REPOSITORY = False
PREPARE_ENVIRONMENT = False
if PREPARE_REPOSITORY:
    if PLATFORM != "colab":
        raise RuntimeError("저장소 준비는 새 Colab용입니다. 로컬 브랜치는 직접 관리하세요.")
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", "--single-branch", "--branch", BRANCH,
                        REPOSITORY_URL, str(REPO_ROOT)], check=True)
    else:
        current = subprocess.check_output(["git", "branch", "--show-current"], cwd=REPO_ROOT, text=True).strip()
        if current != BRANCH:
            raise RuntimeError(f"현재 브랜치가 {current}입니다. 파일 상태를 확인하고 직접 전환하세요.")
        print("기존 저장소를 유지합니다. 원격 업데이트가 필요하면 변경 상태를 확인하고 직접 받으세요.")
if PREPARE_ENVIRONMENT:
    if PLATFORM != "colab" or sys.version_info[:2] != (3, 12):
        raise RuntimeError("별도 Python 3.12 Colab 세션을 준비하세요. 로컬은 uv sync를 직접 실행합니다.")
    if any(name in sys.modules for name in ("torch", "torchvision")):
        raise RuntimeError("torch가 이미 로드됐습니다. 실행 중인 학습을 건드리지 말고 새 세션에서 설치하세요.")
    if shutil.which("uv") is None:
        raise RuntimeError("uv가 없습니다. 저장소 README의 uv 설치를 직접 수행하고 이 셀을 다시 실행하세요.")
    # pylock은 lock의 직접 wheel URL·해시를 보존한다. 실패해도 로그·파일을 지우지 않는다.
    setup_dir = Path(tempfile.mkdtemp(prefix="frcnn_exp3_setup_"))
    pylock = setup_dir / "pylock.toml"
    print("환경 준비 파일·로그:", setup_dir)
    run_setup_command(["uv", "export", "--locked", "--no-dev", "--no-emit-project", "--quiet",
                       "--format", "pylock.toml", "--output-file", str(pylock)],
                      "lock_export", setup_dir)
    run_setup_command(["uv", "pip", "install", "--python", sys.executable, "--require-hashes",
                       "-r", str(pylock)], "package_install", setup_dir)
    _FRCNN_ENVIRONMENT_RESTART_REQUIRED = True
    print("현재 uv.lock 설치 완료. 설치 전 로드된 Pillow·NumPy 등이 남을 수 있습니다.")
    print("이 3차 실험 준비용 커널만 직접 Restart한 뒤, 1번 설정 → 2번 환경 확인 셀 순서로 실행하세요.")
    print("재시작 후 PREPARE_REPOSITORY=False, PREPARE_ENVIRONMENT=False로 두며 설치를 반복하지 않습니다.")

In [ ]:
import importlib.metadata


def check_loaded_packages(installed, modules=None):
    """패키지 재설치 전의 캐시가 남았는지 import 전에 확인한다. 자동 reload는 하지 않는다."""
    modules = sys.modules if modules is None else modules
    names = {"torch": "torch", "torchvision": "torchvision", "numpy": "numpy",
             "pandas": "pandas", "pillow": "PIL", "wandb": "wandb"}
    stale = []
    for package, name in names.items():
        module = modules.get(name)
        version = getattr(module, "__version__", None)
        if version is not None and version != installed[package]:
            stale.append(f"{name}: 메모리={version}, 설치={installed[package]}")
    typing_module = modules.get("PIL._typing")
    if installed["pillow"] == "12.3.0" and typing_module is not None and not hasattr(typing_module, "_Ink"):
        stale.append("PIL._typing: 현재 Pillow 12.3.0이 사용하는 _Ink가 없는 이전 모듈")
    if stale:
        raise RuntimeError(
            "설치 전 모듈이 현재 커널 메모리에 남아 있습니다.\n" + "\n".join(stale) +
            "\n이 3차 실험 준비용 커널만 직접 Restart한 뒤 1번 설정 → 2번 환경 확인 셀을 실행하세요. "
            "설치는 다시 실행하지 않습니다."
        )


if globals().get("_FRCNN_ENVIRONMENT_RESTART_REQUIRED", False):
    raise RuntimeError(
        "이 커널에서 패키지를 설치했습니다. Pillow 등의 이전 모듈을 지우려면 이 준비용 커널을 직접 Restart하세요. "
        "그다음 1번 설정 → 2번 환경 확인 셀을 실행하며 PREPARE_ENVIRONMENT=False로 둡니다."
    )
if not REPO_ROOT.is_dir():
    raise FileNotFoundError("저장소가 없습니다. 새 Colab에서 저장소 준비 셀을 먼저 실행하세요.")
current = subprocess.check_output(["git", "branch", "--show-current"], cwd=REPO_ROOT, text=True).strip()
if current != BRANCH:
    raise RuntimeError(f"브랜치가 {current}입니다. {BRANCH}에서 실행하도록 직접 확인하세요.")
if sys.version_info[:2] != (3, 12):
    raise RuntimeError("Python 3.12 환경을 선택하세요. Python을 자동 교체하지 않습니다.")
LOCK = tomllib.loads((REPO_ROOT / "uv.lock").read_text())
CORE_PACKAGES = ("torch", "torchvision", "numpy", "pandas", "pillow", "pycocotools", "wandb")
INSTALLED_PACKAGES = {}
for package in CORE_PACKAGES:
    locked = {p["version"] for p in LOCK["package"] if p["name"] == package}
    actual = importlib.metadata.version(package)
    INSTALLED_PACKAGES[package] = actual
    if actual not in locked:
        raise RuntimeError(f"{package}={actual}, lock={sorted(locked)}. 현재 uv.lock 환경을 먼저 준비하세요.")
check_loaded_packages(INSTALLED_PACKAGES)
sys.path.insert(0, str(REPO_ROOT / "src"))
os.environ["WANDB_ENTITY"] = "codeit-prj1-team2"
try:
    import PIL
    from PIL import Image, ImageColor, ImageDraw, ImageFont
    import torch
    import torchvision
except (ImportError, OSError, RuntimeError) as error:
    raise RuntimeError(
        f"잠금 환경의 모듈 로드 실패: {type(error).__name__}: {error}\n"
        "설치 뒤 이 준비용 커널을 재시작하지 않았다면 먼저 직접 Restart하세요. "
        "이미 재시작했다면 아래 원본 오류 전체를 확인하고 패키지를 임의 재설치하지 마세요."
    ) from error
check_loaded_packages(INSTALLED_PACKAGES)
print("Python", sys.version.split()[0], "torch", torch.__version__, "torchvision", torchvision.__version__)
print("CUDA", torch.version.cuda, "| GPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")
if PLATFORM == "colab" and not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU를 사용할 수 없습니다. Colab GPU 종류·드라이버와 lock의 CUDA 13.0 wheel 호환을 확인하세요.")

## 3. 공통 함수와 기존 데이터 계약
2차 참고 노트북의 Dataset·라벨 역매핑·이미지 해시·xywh/xyxy 변환을 유지합니다. 새 노트북의 실행 함수와 balance.py, uv.lock도 실험 식별자에 포함합니다.

In [ ]:
"""3차 실험의 공통 도구와 실제 실행 코드 식별자."""
import ast
import contextlib
import copy
import csv
import gc
import hashlib
import importlib.metadata
import inspect
import io
import json
import linecache
import math
import os
import random
import shutil
import subprocess
import sys
import tempfile
import time
import uuid
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from IPython.display import display
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights, fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.ops import FrozenBatchNorm2d, box_convert, box_iou
from torchvision.transforms.functional import to_tensor
from pill_detection.utils import xyxy_to_xywh, draw_boxes
from pill_detection.datasets import OTHERS_CATEGORY_ID
from pill_detection.evaluate import (
    evaluate_val, drop_ignored, load_gt, competition_from_classes, write_submission,
)

NOTEBOOK_CODE_BLOCKS = {}
CORE_PACKAGES = ("torch", "torchvision", "numpy", "pandas", "pillow", "pycocotools", "wandb")


def _register_notebook_block(name):
    """실행한 함수 셀의 소스를 기록한다."""
    frame = inspect.currentframe().f_back
    try:
        source = "".join(linecache.getlines(frame.f_code.co_filename))
        if not source:
            source = get_ipython().history_manager.input_hist_raw[-1]
        if not source.strip():
            raise RuntimeError("실행 셀의 소스를 찾지 못했습니다. 노트북 함수 셀을 위에서부터 다시 실행하세요.")
        NOTEBOOK_CODE_BLOCKS[name] = source
    finally:
        del frame


def file_sha256(path):
    """파일 내용의 SHA-256을 반환한다."""
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def stable_hash(value):
    """경로나 셀 실행 번호와 무관한 JSON 식별자를 만든다."""
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
                                     allow_nan=False).encode()).hexdigest()


def write_json(path, value):
    """완성한 JSON만 교체해 중단 중인 파일을 결과로 사용하지 않는다."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=path.parent,
                                     prefix=path.name + ".", suffix=".part", delete=False) as stream:
        temporary = Path(stream.name)
        json.dump(value, stream, ensure_ascii=False, indent=2, allow_nan=False)
    os.replace(temporary, path)


def write_frame(path, frame):
    """CSV를 임시 파일에서 완성한 뒤 교체한다."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".part")
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def code_identity():
    """실제로 실행한 함수 셀과 팀 공통 코드의 해시를 반환한다."""
    names = ("helpers", "data", "baseline", "sampling", "model", "training", "analysis", "checks")
    missing = set(names) - set(NOTEBOOK_CODE_BLOCKS)
    if missing:
        raise RuntimeError(f"함수 셀이 빠졌습니다: {sorted(missing)}. 함수 셀을 위에서부터 실행하세요.")
    embedded = {name: stable_hash(ast.dump(ast.parse(NOTEBOOK_CODE_BLOCKS[name]),
                                         include_attributes=False)) for name in names}
    common = {name: file_sha256(REPO_ROOT / "src/pill_detection" / name)
              for name in ("utils.py", "data.py", "datasets.py", "evaluate.py", "balance.py")}
    return {"notebook": embedded, "common": common}


def package_versions():
    """재개 비교용 기본 버전을 반환하고 CUDA 빌드 접미사는 별도로 기록한다."""
    return {name: importlib.metadata.version(name).split("+")[0] for name in CORE_PACKAGES}


def runtime_info():
    """실행 환경은 재개 식별자와 구분해 이력으로 기록한다."""
    result = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT,
                            capture_output=True, text=True, check=True)
    dirty = subprocess.run(["git", "status", "--porcelain"], cwd=REPO_ROOT,
                           capture_output=True, text=True, check=True)
    return {"time_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "platform": PLATFORM, "python": sys.version.split()[0],
            "python_executable": sys.executable, "packages": {
                name: importlib.metadata.version(name) for name in CORE_PACKAGES},
            "cuda": torch.version.cuda,
            "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
            "repository_commit": result.stdout.strip(), "repository_dirty": bool(dirty.stdout.strip()),
            "lock_sha256": file_sha256(REPO_ROOT / "uv.lock"),
            "cuda_allocator_config": os.environ.get("PYTORCH_ALLOC_CONF") or os.environ.get("PYTORCH_CUDA_ALLOC_CONF"),
            "deterministic_algorithms": torch.are_deterministic_algorithms_enabled()}


def save_figure(path, figure):
    """그림을 저장하고 노트북에 표시한 뒤 메모리를 반환한다."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(path, dpi=150, bbox_inches="tight")
    display(figure)
    plt.close(figure)


_register_notebook_block("helpers")

In [ ]:
"""팀에서 생성한 COCO 데이터와 원래 K-코드 평가 정답을 연결한다."""
IMAGE_HASH_CACHE = {}
EXPECTED_DATA = {"v2": (7129, 118), "v3": (8974, 96), "v4": (11206, 75)}
BUNDLES = {}


def image_digest(path):
    """같은 세션에서 동일한 이미지 내용을 여러 방법이 다시 읽지 않도록 캐시한다."""
    path = Path(path)
    stat = path.stat()
    key = (str(path.resolve()), stat.st_size, stat.st_mtime_ns)
    if key not in IMAGE_HASH_CACHE:
        IMAGE_HASH_CACHE[key] = file_sha256(path)
    return IMAGE_HASH_CACHE[key]


class Exp2Dataset(Dataset):
    """COCO categories의 idx+1을 배경 0과 분리해 torchvision에 전달한다."""

    def __init__(self, image_dir, annotation_path, label_of):
        self.image_dir = Path(image_dir)
        self.doc = json.loads(Path(annotation_path).read_text(encoding="utf-8"))
        self.frame = pd.DataFrame(self.doc["images"]).rename(columns={"id": "image_id"})
        self.frame = self.frame.sort_values("image_id").reset_index(drop=True)
        if self.frame["file_name"].duplicated().any() or self.frame["image_id"].duplicated().any():
            raise ValueError("이미지 이름·ID가 중복됩니다. 팀 학습 폴더를 다시 확인하세요.")
        self.annotations = defaultdict(list)
        self.label_of = label_of
        image_ids = set(self.frame["image_id"])
        for ann in self.doc["annotations"]:
            if ann["image_id"] not in image_ids or ann["category_id"] not in label_of:
                raise ValueError("COCO annotation의 이미지·클래스 연결이 다릅니다. 학습 폴더를 확인하세요.")
            self.annotations[ann["image_id"]].append(ann)
        for row in self.frame.itertuples():
            if Path(row.file_name).name != row.file_name or not (self.image_dir / row.file_name).is_file():
                raise FileNotFoundError(f"학습 이미지가 없습니다: {row.file_name}. ZIP 전개와 데이터 경로를 확인하세요.")

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(self.image_dir / row.file_name) as image:
            if image.size != (int(row.width), int(row.height)):
                raise ValueError(f"이미지 크기가 COCO 정답과 다릅니다: {row.file_name}. 데이터를 확인하세요.")
            tensor = to_tensor(image.convert("RGB"))
        anns = self.annotations[int(row.image_id)]
        boxes = torch.tensor([ann["bbox"] for ann in anns], dtype=torch.float32).reshape(-1, 4)
        if boxes.numel() and (not torch.isfinite(boxes).all() or (boxes[:, 2:] <= 0).any()):
            raise ValueError(f"유효하지 않은 xywh bbox입니다: {row.file_name}. 학습 폴더를 확인하세요.")
        areas = boxes[:, 2] * boxes[:, 3]
        return tensor, {"boxes": box_convert(boxes, "xywh", "xyxy"),
                        "labels": torch.tensor([self.label_of[a["category_id"]] for a in anns], dtype=torch.int64),
                        "image_id": torch.tensor(int(row.image_id)), "area": areas,
                        "iscrowd": torch.tensor([a.get("iscrowd", 0) for a in anns], dtype=torch.int64)}


class TestImages(Dataset):
    """정답 없는 test 이미지도 누락 없이 manifest에 남긴다."""

    def __init__(self, image_dir):
        self.image_dir = Path(image_dir)
        rows = []
        for path in sorted(self.image_dir.glob("*.png"), key=lambda p: p.name):
            if not path.stem.isdigit():
                raise ValueError("Test 파일명은 숫자.png여야 합니다. 대회 test 폴더를 확인하세요.")
            with Image.open(path) as image:
                rows.append({"file_name": path.name, "image_id": int(path.stem),
                             "width": image.width, "height": image.height})
        self.frame = pd.DataFrame(rows)
        if self.frame.empty or self.frame["image_id"].duplicated().any():
            raise ValueError("Test 이미지가 없거나 ID가 중복됩니다. test_images.zip과 경로를 확인하세요.")
        self.frame = self.frame.sort_values("image_id").reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(self.image_dir / row.file_name) as image:
            tensor = to_tensor(image.convert("RGB"))
        return tensor, {"image_id": torch.tensor(int(row.image_id))}


def collate_detection(batch):
    """이미지와 객체 수가 달라도 작은 배치로 묶는다."""
    return tuple(zip(*batch))


def prepare_bundle(variant, expected=True):
    """방법별 학습 데이터와 변하지 않는 공식 평가 정답을 한 번 준비한다."""
    if variant in BUNDLES:
        return BUNDLES[variant]
    root = DATASET_ROOT / variant
    required = [root / name for name in ("coco/train.json", "coco/val.json", "eval/val_gt.json",
                                         "classes.csv", "dataset_info.json")]
    if any(not p.is_file() for p in required):
        raise FileNotFoundError(f"{variant} 학습 폴더가 없습니다. 데이터 생성 셀을 먼저 실행하세요.")
    info = json.loads((root / "dataset_info.json").read_text())
    if info.get("variant") != variant:
        raise ValueError("dataset_info의 방법이 다릅니다. 학습 폴더를 확인하세요.")
    if expected:
        for name, key in (("images.csv", "images_csv_sha256"), ("classes.csv", "classes_csv_sha256")):
            if info.get(key) != file_sha256(REPO_ROOT / "data/datasets" / name):
                raise ValueError("학습 폴더가 현재 팀 목록과 다릅니다. 데이터를 다시 준비하세요.")
    cats = json.loads((root / "coco/train.json").read_text())["categories"]
    if sorted(c["idx"] for c in cats) != list(range(len(cats))):
        raise ValueError("COCO categories.idx가 연속적이지 않습니다. 팀 생성 데이터를 확인하세요.")
    label_of = {int(c["id"]): int(c["idx"]) + 1 for c in cats}
    if len(label_of) != len(cats):
        raise ValueError("중복 클래스 ID입니다. 팀 생성 데이터를 확인하세요.")
    datasets = {split: Exp2Dataset(root / "images" / split, root / "coco" / f"{split}.json", label_of)
                for split in ("train", "val")}
    if datasets["val"].doc["categories"] != cats:
        raise ValueError("Train·val 클래스 매핑이 다릅니다. 학습 폴더를 확인하세요.")
    gt = load_gt(root / "eval/val_gt.json")
    if set(datasets["train"].frame.file_name) & set(datasets["val"].frame.file_name):
        raise ValueError("Train과 val 이미지가 겹칩니다. 팀 목록을 확인하세요.")
    if set(datasets["val"].frame.file_name) != {image["file_name"] for image in gt["images"]}:
        raise ValueError("학습용 val과 공식 정답 이미지가 다릅니다. 같은 방법의 학습 폴더를 확인하세요.")
    if expected and (len(datasets["train"]), len(cats), len(gt["images"]), len(gt["annotations"])) != (
            *EXPECTED_DATA[variant], 1000, 3790):
        raise ValueError("팀 안내의 데이터 개수와 다릅니다. --check와 데이터 전개 상태를 확인하세요.")
    for other in BUNDLES.values():
        if stable_hash(other["gt"]) != stable_hash(gt):
            raise ValueError("방법 사이의 공식 val 정답이 다릅니다. 팀 데이터 목록을 확인하세요.")
    print(f"{variant}: 이미지 내용 해시 확인 중(세션에서 같은 이미지는 재사용)…", flush=True)
    image_hashes = {split: {r.file_name: image_digest(ds.image_dir / r.file_name)
                             for r in ds.frame.itertuples()} for split, ds in datasets.items()}
    if set(image_hashes["train"].values()) & set(image_hashes["val"].values()):
        raise ValueError("내용이 같은 이미지가 train/val에 겹칩니다. 데이터 담당자에게 확인하세요.")
    fingerprint = {"variant": variant, "files": {p.relative_to(root).as_posix(): file_sha256(p)
                    for p in required if p.name != "dataset_info.json"}, "images": image_hashes}
    mapping = pd.DataFrame([{"category_id": int(c["id"]), "model_label": label_of[c["id"]],
                             "k_code": str(c.get("k_code", c["id"])),
                             "name": str(c.get("product_name", c.get("name", c["id"])))} for c in cats])
    bundle = {"variant": variant, "root": root, "datasets": datasets, "gt": gt,
              "class_map": mapping, "category_to_label": label_of,
              "label_to_category": {v: k for k, v in label_of.items()},
              "identity": stable_hash(fingerprint), "dataset_info": info, "fingerprint": fingerprint}
    BUNDLES[variant] = bundle
    return bundle


def make_dataset(bundle, split):
    """학습과 validation loss에는 방법별 COCO 라벨을 사용한다."""
    return bundle["datasets"][split]


def show_data(bundle):
    """개수·클래스 연결과 실제 정답 bbox를 보여준다."""
    rows = [{"split": split, "images": len(ds), "boxes": len(ds.doc["annotations"])}
            for split, ds in bundle["datasets"].items()]
    display(pd.DataFrame(rows))
    display(bundle["class_map"])
    dataset = make_dataset(bundle, "train")
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for axis, index in zip(axes, range(min(3, len(dataset)))):
        image, target = dataset[index]
        labels = [str(bundle["label_to_category"][int(v)]) for v in target["labels"]]
        picture = draw_boxes(dataset.image_dir / dataset.frame.iloc[index].file_name,
                             [xyxy_to_xywh(b.tolist()) for b in target["boxes"]], labels, color="lime")
        axis.imshow(picture); axis.axis("off")
    plt.show(); plt.close(fig)


def predictions_for_team(records):
    """추론 캐시를 팀 공식 함수의 K-코드·xywh 형식으로 변환한다."""
    return [{"file_name": row["file_name"], "category_id": int(label),
             "bbox": xyxy_to_xywh(box.tolist()), "score": float(score)}
            for row in records for box, label, score in zip(
                row["prediction"]["boxes"], row["prediction"]["labels"], row["prediction"]["scores"])]


_register_notebook_block("data")

In [ ]:
BASELINE_FILE_SHA256 = {
    "config.json": "17b91b5be8fbb36a906c3f9d246ed129dd68fb49382c7ecbc40a26916600fb41",
    "dataset_info.json": "6b38873bb187cdd5c0e352074693ba877c07f2897ff610448c1ffe4a94a88ae8",
    "result.json": "4c503738e75069e7c15c945e96aba3981e3cf64ca53491193d41ec1aa135fa69",
    "scores.json": "438e9c0bafd4230313f753c33085848f97a952a029cb5e2a5f7e00ddb0e42f9b",
    "metrics.csv": "c9e1089e331a4a67ffee255de911aa2ab04af8e857b3b42b8fe2758015cfadf5"
}
BASELINE_BEST_SHA256 = "1450c8d4c8581f7035f865f5e25ff8dd2f1dc9de1860842132711b028d87d553"

def load_baseline():
    """지정 run의 설정·예측·best를 대조하고 원본은 읽기만 한다."""
    docs = {}
    for name, expected in BASELINE_FILE_SHA256.items():
        p = BASELINE_DIR / name
        if not p.is_file() or file_sha256(p) != expected:
            raise ValueError(f"baseline의 {name}이 확인한 원본과 다릅니다. rw_runs/v3_frcnn_e10_s0을 확인하세요.")
        docs[name] = json.loads(p.read_text()) if name.endswith(".json") else pd.read_csv(p)
    manifest, result, info = (docs[n] for n in ("config.json", "result.json", "dataset_info.json"))
    if manifest["spec"]["run_name"] != "v3_frcnn_e10_s0" or manifest["spec"]["epochs"] != 10:
        raise ValueError("기준 run이 v3 e10이 아닙니다. baseline 경로를 확인하세요.")
    if stable_hash(manifest["recipe"]) != manifest["signature"] or result["signature"] != manifest["signature"]:
        raise ValueError("baseline 설정 서명이 일치하지 않습니다. 원본 설정 파일을 확인하세요.")
    if info["n_images"] != {"train": 8974, "val": 1000} or info["nc"] != 96:
        raise ValueError("baseline 데이터 수가 다릅니다. 지정한 v3 run을 확인하세요.")
    for name, key in (("images.csv", "images_csv_sha256"), ("classes.csv", "classes_csv_sha256")):
        if file_sha256(REPO_ROOT / "data/datasets" / name) != info[key]:
            raise ValueError(f"고정 목록 {name}이 baseline과 다릅니다. CSV를 고치지 말고 버전을 확인하세요.")
    for name, expected in manifest["recipe"]["code"]["common"].items():
        if file_sha256(REPO_ROOT / "src/pill_detection" / name) != expected:
            raise ValueError(f"공용 {name}이 baseline과 다릅니다. 데이터·평가 구현을 확인하세요.")
    best = BASELINE_DIR / "weights/best.pt"
    if not best.is_file() or file_sha256(best) != BASELINE_BEST_SHA256:
        raise ValueError("baseline best.pt가 확인한 파일과 다릅니다. 지정한 run의 best를 준비하세요.")
    fp = json.loads((BASELINE_DIR / "data_fingerprint.json").read_text())
    if stable_hash(fp) != manifest["recipe"]["data_identity"]:
        raise ValueError("baseline 데이터 지문이 설정과 다릅니다. 원본 파일을 확인하세요.")
    meta = json.loads((BASELINE_DIR / "val_prediction_identity.json").read_text())
    identity = stable_hash({"checkpoint": BASELINE_BEST_SHA256, "images": fp["images"]["val"],
                            "signature": manifest["signature"], "split": "val"})
    if meta["identity"] != identity or meta["checkpoint_epoch"] != result["best_epoch"]:
        raise ValueError("baseline best와 val 예측의 연결이 다릅니다. 예측 캐시를 확인하세요.")
    for name, key in (("val_predictions.json", "predictions_sha256"), ("val_manifest.csv", "manifest_sha256")):
        if file_sha256(BASELINE_DIR / name) != meta[key]:
            raise ValueError(f"baseline {name} 해시가 다릅니다. 원본 캐시를 확인하세요.")
    if file_sha256(BASELINE_DIR / "val_gt.json") != fp["files"]["eval/val_gt.json"]:
        raise ValueError("baseline val 정답 해시가 다릅니다. 원본을 확인하세요.")
    history = docs["metrics.csv"]
    packages = package_versions()
    difference = {k: {"baseline": v, "current": packages.get(k)} for k, v in manifest["recipe"]["packages"].items()
                  if packages.get(k) != v}
    return {"manifest": manifest, "result": result, "info": info, "scores": docs["scores.json"],
            "metrics": history, "fingerprint": fp, "val_metadata": meta,
            "best_sha256": BASELINE_BEST_SHA256, "file_sha256": BASELINE_FILE_SHA256,
            "actual_optimizer_updates": int(history.optimizer_steps.sum()),
            "amp_skipped_steps": int(history.amp_skipped_steps.sum()),
            "package_differences": difference, "environment_confounded": bool(difference),
            "actual_public_score": BASELINE_PUBLIC_SCORE, "public_score_source": "사용자 제공 제출 기록"}


def baseline_snapshot(baseline):
    """새 run에 보관할 작은 기준 정보. 가중치·원본 결과는 복사하거나 수정하지 않는다."""
    return {k: v for k, v in baseline.items() if k not in ("metrics", "fingerprint")}


_register_notebook_block("baseline")

In [ ]:
import statistics
from pill_detection.balance import (
    load_train_objects, repeat_list, write_train_list, class_weights,
    use_fastrcnn_class_weights, class_ap_rows, imbalance_summary,
)


def epoch_budget(n_base, n_list, epochs=10, batch=16, accumulation=1, warmup_epochs=3):
    """마지막 불완전 배치를 포함해 예정 optimizer 갱신과 warmup을 환산한다."""
    if min(n_base, n_list, batch, accumulation) <= 0:
        raise ValueError("학습 목록과 배치 크기는 양수여야 합니다. 데이터 준비 셀을 확인하세요.")
    scale = n_base / n_list
    count = round(epochs * scale)
    steps = math.ceil(math.ceil(n_list / batch) / accumulation)
    base_steps = math.ceil(math.ceil(n_base / batch) / accumulation)
    warm = warmup_epochs * scale
    return {"n_base": n_base, "n_list": n_list, "epochs": count, "scale": scale,
            "steps_per_epoch": steps, "total_optimizer_updates_planned": count * steps,
            "baseline_total_updates_planned": epochs * base_steps,
            "updates_difference": count * steps - epochs * base_steps,
            "updates_ratio": count * steps / (epochs * base_steps),
            "warmup_epochs": warm, "warmup_steps": int(warm * steps),
            "baseline_warmup_steps": int(warmup_epochs * base_steps),
            "last_batch_images": (n_list - 1) % batch + 1, "drop_last": False}


def make_plans(bundle, baseline):
    """공용 RFS 목록·가중치와 고정 v3 기준의 세 독립 학습 계획을 만든다."""
    if tuple(CONDITIONS) != ("rfs", "weight", "both") or WEIGHT_POWER != 0.5 or RFS_SEED != 0:
        raise ValueError("확정한 세 조건·power 0.5·seed 0과 다릅니다. 1번 설정을 확인하세요.")
    if RFS_THRESHOLD is not None:
        raise ValueError("이번 실험은 팀 기본 threshold를 사용합니다. RFS_THRESHOLD=None으로 두세요.")
    if bundle["identity"] != baseline["manifest"]["recipe"]["data_identity"]:
        raise ValueError("현재 v3 데이터 지문이 baseline과 다릅니다. 원본 데이터·정제 사본을 확인하세요.")
    if bundle["category_to_label"] != {int(k): v for k, v in baseline["manifest"]["recipe"]["class_mapping"].items()}:
        raise ValueError("클래스 매핑이 baseline과 다릅니다. 같은 v3 학습 폴더를 준비하세요.")
    objects = load_train_objects(bundle["root"])
    rfs = repeat_list(objects, threshold=RFS_THRESHOLD, seed=RFS_SEED)
    if len(objects) != 8974 or len(rfs) != 10148 or set(rfs) != set(objects):
        raise ValueError("v3 RFS 목록이 확인한 8974→10148과 다릅니다. 데이터·balance.py 버전을 확인하세요.")
    image_counts = Counter(c for obj in objects.values() for c in set(obj["classes"]))
    threshold = statistics.median(n / len(objects) for n in image_counts.values())
    base_files = bundle["datasets"]["train"].frame.file_name.tolist()
    sampling_path = bundle["root"] / "sampling/rfs/train.txt"
    expected_text = "".join(f"{(bundle['root'] / 'images/train' / name).resolve()}\n" for name in rfs)
    if sampling_path.exists():
        if sampling_path.read_text() != expected_text:
            raise ValueError("기존 RFS 목록이 다릅니다. 덮어쓰지 말고 새로운 데이터 준비 경로를 지정하세요.")
    else:
        write_train_list(bundle["root"], "rfs", rfs)
    base_counts = Counter(c for obj in objects.values() for c in obj["classes"])
    categories = sorted(bundle["datasets"]["train"].doc["categories"], key=lambda c: c["idx"])
    plans = {}
    for condition in CONDITIONS:
        files = rfs if condition in ("rfs", "both") else base_files
        power = WEIGHT_POWER if condition in ("weight", "both") else 0.0
        counts = Counter(c for name in files for c in objects[name]["classes"])
        weights = class_weights(bundle["root"], power, files) if power else [1.0] * len(categories)
        budget = epoch_budget(len(objects), len(files))
        if budget["epochs"] != (10 if condition == "weight" else 9):
            raise ValueError("계산한 epoch가 확정 계획과 다릅니다. 목록과 설정을 확인하세요.")
        config = copy.deepcopy(baseline["manifest"]["config"])
        config.update(max_epochs=budget["epochs"], warmup_epochs=budget["warmup_epochs"], num_workers=8)
        spec = {"variant": "v3", "epochs": budget["epochs"], "seed": 0, "mode": "ft", "lr": 3e-4,
                "condition": condition, "run_name": f"v3_frcnn_exp3_{condition}_e{budget['epochs']}_s0" + RUN_SUFFIX}
        if Path(spec["run_name"]).name != spec["run_name"]:
            raise ValueError("RUN_SUFFIX에 경로 문자가 있습니다. 파일 이름 구분자만 사용하세요.")
        frequency = pd.DataFrame([{"category_id": int(c["id"]), "model_label": int(c["idx"]) + 1,
            "k_code": f"K-{int(c['id']):06d}", "n_train_original": base_counts[c["id"]],
            "images_with_class": image_counts[c["id"]], "image_frequency": image_counts[c["id"]] / len(objects),
            "class_repeat_factor": (max(1.0, math.sqrt(threshold / (image_counts[c["id"]] / len(objects))))
                                    if condition in ("rfs", "both") else 1.0),
            "expected_objects_per_epoch": (sum(max(1.0, max(math.sqrt(threshold / (image_counts[k] / len(objects)))
                for k in obj["classes"])) * obj["classes"].count(c["id"]) for obj in objects.values())
                if condition in ("rfs", "both") else base_counts[c["id"]]),
            "fixed_list_objects_per_epoch": counts[c["id"]], "class_weight": weights[i],
            "monitor": int(c["id"]) in MONITOR_CLASSES} for i, c in enumerate(categories)])
        background = sum(w * counts[c["id"]] for w, c in zip(weights, categories)) / sum(counts.values())
        sampling = {"method": "fixed_rfs" if condition != "weight" else "original",
                    "threshold": threshold if condition != "weight" else None, "seed": RFS_SEED,
                    "weight_power": power, "weight_cap": None, "background_weight": background,
                    "files_sha256": stable_hash(files), "n_unique": len(set(files)), **budget}
        plans[condition] = {"spec": spec, "config": config, "sampling": sampling, "files": files,
                            "frequency": frequency, "counts": counts}
    return plans


def training_subset(bundle, plan):
    """중복을 제거하지 않고 공용 목록의 각 줄을 Dataset index로 연결한다."""
    dataset = bundle["datasets"]["train"]
    index_of = {name: i for i, name in enumerate(dataset.frame.file_name)}
    return Subset(dataset, [index_of[name] for name in plan["files"]])


def check_condition(bundle, plan):
    """실제 전역 RoI loss의 가중치와 Dataset 길이가 해당 조건인지 확인한다."""
    from torchvision.models.detection import roi_heads
    weight = getattr(roi_heads.fastrcnn_loss, "weight", None)
    weighted = bool(plan["sampling"]["weight_power"])
    if weighted != (weight is not None):
        raise RuntimeError("분류 가중치 활성화 상태가 조건과 다릅니다. 조건 적용 셀을 확인하세요.")
    if weighted:
        expected = torch.tensor([plan["sampling"]["background_weight"], *plan["frequency"].class_weight])
        if not torch.allclose(weight.cpu(), expected, rtol=1e-5, atol=1e-6):
            raise RuntimeError("실제 클래스·배경 가중치가 계획과 다릅니다. 샘플링 후 객체 수를 확인하세요.")
    if len(training_subset(bundle, plan)) != plan["sampling"]["n_list"]:
        raise RuntimeError("실제 Dataset 길이가 목록과 다릅니다. 반복 index를 확인하세요.")
    return {"condition": plan["spec"]["condition"], "weight_applied": weighted,
            "weight_vector": weight.tolist() if weighted else None,
            "n_train_list": plan["sampling"]["n_list"], "files_sha256": plan["sampling"]["files_sha256"]}


_register_notebook_block("sampling")

In [ ]:
FIXED_CONFIG = {"max_epochs":10,"patience":30,"image_size":640,"batch_size":16,"accumulation_steps":1,"weight_decay":0.0001,"betas":[0.9,0.999],"warmup_epochs":3,"final_lr_ratio":0.01,"score_threshold":0.001,"nms_iou":0.5,"max_detections":100,"preview_score":0.3,"preview_count":3,"wandb_project":"pill-object-detection","wandb_entity":"codeit-prj1-team2","wandb_mode":"online","use_wandb":True,"require_cuda":True,"deterministic":False,"precision":"amp_fp16","normalization":"FrozenBatchNorm2d_eps0_fixed_statistics_and_affine","augmentation":"none","weights":"COCO_V1","optimizer":"AdamW","learning_rate":0.0003,"learning_rates":[0.0003],"val_loss_batch_size":8,"inference_batch_size":8,"loss_measurement_seed":0,"inference_precision":"fp32","val_loss_precision":"fp32","num_workers":8}


def validate_config(config):
    """baseline의 실제 고정 설정과 승인한 epoch/warmup만 허용한다."""
    if set(config) != set(FIXED_CONFIG):
        raise ValueError("설정 키가 baseline과 다릅니다. 계획 셀에서 만든 config를 사용하세요.")
    for key, value in FIXED_CONFIG.items():
        if key not in ("max_epochs", "warmup_epochs") and config[key] != value:
            raise ValueError(f"고정 설정 {key}가 다릅니다. baseline 설정으로 되돌리세요.")
    if config["max_epochs"] not in (9, 10) or not 0 < config["warmup_epochs"] <= 3:
        raise ValueError("epoch/warmup이 확정 계획과 다릅니다. 계획 셀을 확인하세요.")


def validate_spec(spec):
    """이번 작업의 v3 Faster R-CNN 세 조건만 허용한다."""
    expected_epochs = 10 if spec.get("condition") == "weight" else 9
    if (spec.get("condition") not in CONDITIONS or spec.get("epochs") != expected_epochs or
        spec.get("variant") != "v3" or spec.get("seed") != 0 or spec.get("mode") != "ft" or spec.get("lr") != 3e-4):
        raise ValueError("v3·세 조건·seed 0·독립 COCO 초기화 계획을 확인하세요.")


def _loader(dataset, config, training=False, generator=None):
    """학습 16·추론 8을 실제 DataLoader에 적용하며 마지막 배치를 남긴다."""
    return DataLoader(dataset, batch_size=config["batch_size"] if training else config["inference_batch_size"],
                      shuffle=training, drop_last=False, num_workers=config["num_workers"],
                      collate_fn=collate_detection, generator=generator, pin_memory=torch.cuda.is_available())


def seed_everything(seed, deterministic=True):
    """분할을 변경하지 않고 학습 난수만 설정한다.

    Args:
        seed (int): 학습 seed.
        deterministic (bool): 결정적 연산 사용 여부.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = deterministic
    torch.use_deterministic_algorithms(deterministic)


def _frozen_normalization(module):
    for name, child in list(module.named_children()):
        if isinstance(child, torch.nn.BatchNorm2d):
            frozen = FrozenBatchNorm2d(child.num_features, eps=0.0)
            for attribute in ("weight", "bias", "running_mean", "running_var"):
                getattr(frozen, attribute).copy_(getattr(child, attribute).detach())
            setattr(module, name, frozen)
        elif isinstance(child, FrozenBatchNorm2d):
            child.eps = 0.0
        else:
            _frozen_normalization(child)


def build_model(num_classes, spec, config, load_coco=True):
    """같은 COCO 검출 모델에서 출력층과 학습 범위를 설정한다.

    Args:
        num_classes (int): 배경을 제외한 클래스 수.
        spec (dict): 실험 조건.
        config (dict): 공통 설정.
        load_coco (bool): 신규 학습은 True. 저장된 전체 체크포인트 복원 시에만 False.

    Returns:
        Module: CPU의 Faster R-CNN. 모든 정규화 통계·계수는 고정.
    """
    validate_config(config)
    validate_spec(spec)
    weights = FasterRCNN_ResNet50_FPN_Weights.COCO_V1 if load_coco else None
    model = fasterrcnn_resnet50_fpn(weights=weights, weights_backbone=None,
                                    trainable_backbone_layers=5 if load_coco else None,
                                    min_size=config["image_size"], max_size=config["image_size"])
    _frozen_normalization(model)
    features = model.roi_heads.box_predictor.cls_score.in_features
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(spec["seed"])
        model.roi_heads.box_predictor = FastRCNNPredictor(features, num_classes + 1)
    for parameter in model.parameters():
        parameter.requires_grad_(True)
    model.roi_heads.score_thresh = config["score_threshold"]
    model.roi_heads.nms_thresh = config["nms_iou"]
    model.roi_heads.detections_per_img = config["max_detections"]
    return model


def make_optimizer(model, spec, config):
    """고정 파라미터를 제외하고 AdamW decay 그룹을 만든다.

    Args:
        model (Module): 학습 모델.
        spec (dict): 학습률 조건.
        config (dict): AdamW 설정.

    Returns:
        AdamW: bias·1차원 파라미터에는 decay 미적용.
    """
    groups = [[], []]
    for name, parameter in model.named_parameters():
        if parameter.requires_grad:
            groups[int(name.endswith("bias") or parameter.ndim <= 1)].append(parameter)
    return torch.optim.AdamW([{"params": groups[0], "weight_decay": config["weight_decay"]},
                               {"params": groups[1], "weight_decay": 0.0}],
                              lr=spec["lr"], betas=tuple(config["betas"]))


def make_scheduler(optimizer, steps_per_epoch, config):
    """optimizer step 단위의 선형 warmup·cosine 스케줄러를 만든다.

    Args:
        optimizer (Optimizer): AdamW.
        steps_per_epoch (int): 누적 후 epoch당 예정 optimizer 갱신 수.
        config (dict): 학습 예산·warmup 설정.

    Returns:
        LambdaLR: 첫 step부터 warmup을 적용하는 scheduler.
    """
    total = config["max_epochs"] * steps_per_epoch
    warmup = int(config["warmup_epochs"] * steps_per_epoch)

    def factor(step):
        if warmup and step < warmup:
            return (step + 1) / warmup
        progress = min(1.0, max(0.0, (step-warmup) / max(1, total-warmup-1)))
        return config["final_lr_ratio"] + (1-config["final_lr_ratio"]) * (1+math.cos(math.pi*progress))/2

    return torch.optim.lr_scheduler.LambdaLR(optimizer, factor)


def _synchronize(device):
    if device.type == "cuda":
        torch.cuda.synchronize()


def collect_predictions(model, dataset, config, reverse, with_targets=True):
    """추론을 한 번 수행해 원본 category ID의 공통 결과를 만든다.

    Args:
        model (Module): 복원 또는 학습 모델.
        dataset (PillDataset): 순서가 고정된 이미지 목록.
        config (dict): 추론 설정.
        reverse (dict): 모델 번호→원래 category ID.
        with_targets (bool): validation 정답 포함 여부.

    Returns:
        tuple: 공통 결과, 순수 추론 시간·전체 시간.
    """
    device = next(model.parameters()).device
    records, model_seconds = [], 0.0
    start_all = time.perf_counter()
    model.eval()
    with torch.inference_mode():
        for images, targets in _loader(dataset, config):
            inputs = [image.to(device) for image in images]
            _synchronize(device)
            start = time.perf_counter()
            outputs = model(inputs)
            _synchronize(device)
            model_seconds += time.perf_counter()-start
            for output, target in zip(outputs, targets):
                info = dataset.frame.iloc[len(records)].to_dict()
                prediction = {key: output[key].detach().cpu() for key in ("boxes", "labels", "scores")}
                prediction["labels"] = torch.tensor([reverse[int(c)] for c in prediction["labels"]], dtype=torch.int64)
                record = {key: int(info[key]) for key in ("image_id", "width", "height")}
                record.update(file_name=info["file_name"], image_path=str(dataset.image_dir / info["file_name"]),
                              prediction=prediction)
                if with_targets:
                    record["target"] = {"boxes": target["boxes"].cpu(),
                                        "labels": torch.tensor([reverse[int(c)] for c in target["labels"]], dtype=torch.int64)}
                records.append(record)
    return records, {"model_ms_per_image": 1000*model_seconds/max(1, len(records)),
                     "wall_seconds": time.perf_counter()-start_all, "images": len(records)}


def _make_scaler(config, device):
    """CUDA AMP 설정에 맞는 손실 스케일러를 만든다.

    Args:
        config (dict): 정밀도 설정.
        device (torch.device): 학습 장치.

    Returns:
        GradScaler: FP32에서는 비활성화한 스케일러.
    """
    enabled = config["precision"] == "amp_fp16"
    if enabled and device.type != "cuda":
        raise RuntimeError("AMP FP16 학습에는 CUDA가 필요합니다. 현재 환경의 CUDA GPU를 선택하세요.")
    return torch.amp.GradScaler("cuda", enabled=enabled)


def _batch_policy(config):
    """실제 배치와 누적 후 목표 배치를 구분해 기록한다.

    Args:
        config (dict): 실제 배치·누적 횟수 설정.

    Returns:
        dict: 단일 GPU 기준 실제 배치·누적 횟수·목표 실효 배치.
    """
    return {"micro_batch_size": config["batch_size"],
            "accumulation_steps": config["accumulation_steps"],
            "effective_batch_size": config["batch_size"] * config["accumulation_steps"]}


def _accumulation_groups(loader, steps):
    """CPU의 작은 배치를 묶으며 마지막 불완전한 묶음도 반환한다.

    Args:
        loader (DataLoader): 원본 CPU 이미지와 정답을 읽는 로더.
        steps (int): 가중치 갱신 전 누적할 최대 작은 배치 수.

    Yields:
        list: GPU로 한 번에 올리지 않을 작은 배치 목록.
    """
    group = []
    for batch in loader:
        group.append(batch)
        if len(group) == steps:
            yield group
            group = []
    if group:
        yield group


def _training_step(model, optimizer, scaler, batches, config):
    """작은 배치별 역전파 후 누적 묶음마다 한 번만 가중치를 갱신한다.

    Args:
        model (Module): 학습 모드의 검출 모델.
        optimizer (Optimizer): 학습 optimizer.
        scaler (GradScaler): 묶음 전체에서 같은 scale을 유지할 스케일러.
        batches (list): CPU 이미지·정답의 작은 배치 목록.
        config (dict): 학습 정밀도·누적 횟수 설정.

    Returns:
        tuple: 실제 이미지 수로 가중 평균한 세부 loss와 갱신 성공 여부.
    """
    if not batches or len(batches) > config["accumulation_steps"]:
        raise ValueError("누적 묶음 크기가 잘못됐습니다. accumulation_steps와 로더를 확인하세요.")
    total_images = sum(len(images) for images, _ in batches)
    if total_images == 0:
        raise ValueError("빈 학습 묶음입니다. 고정 학습 분할을 확인하세요.")
    device = next(model.parameters()).device
    optimizer.zero_grad(set_to_none=True)
    means = {}
    for images, targets in batches:
        if not images or len(images) != len(targets) or len(images) > config["batch_size"]:
            raise ValueError("작은 배치의 이미지·정답 수가 잘못됐습니다. batch_size와 로더를 확인하세요.")
        inputs = [image.to(device) for image in images]
        truth = [{key: value.to(device) for key, value in target.items()} for target in targets]
        weight = len(images) / total_images
        with torch.autocast(device_type=device.type, dtype=torch.float16,
                            enabled=config["precision"] == "amp_fp16"):
            losses = model(inputs, truth)
            # 마지막 묶음이 16개 미만이어도 실제 이미지 수로 평균한다.
            # 검출 loss의 내부 정규화·RoI 샘플링 때문에 물리 배치 16과 완전히 같지는 않다.
            loss = sum(losses.values()) * weight
        if not torch.isfinite(loss):
            raise FloatingPointError("학습 loss가 유한하지 않습니다. 이미지·라벨·학습률을 확인하세요.")
        scaler.scale(loss).backward()
        for key, value in losses.items():
            means[key] = means.get(key, 0.0) + float(value.detach()) * weight
        # 이전 작은 배치의 GPU 입력과 그래프를 다음 forward까지 붙잡지 않는다.
        del inputs, truth, losses, loss
    old_scale = scaler.get_scale()
    scaler.step(optimizer)
    scaler.update()
    updated = not scaler.is_enabled() or scaler.get_scale() >= old_scale
    return means, updated


@contextlib.contextmanager
def _validation_loss_context(model, seed):
    """검출 loss 경로만 켜고 모드·난수·정규화 버퍼를 보존한다.

    Args:
        model (Module): 현재 epoch의 모델.
        seed (int): RPN·RoI 표본 추출에 사용할 고정 seed.
    """
    modes = {module: module.training for module in model.modules()}
    buffers = {name: value.detach().clone() for name, value in model.named_buffers()}
    python_state, numpy_state = random.getstate(), np.random.get_state()
    device = next(model.parameters()).device
    devices = [device.index if device.index is not None else torch.cuda.current_device()] if device.type == "cuda" else []
    try:
        with torch.random.fork_rng(devices=devices):
            random.seed(seed)
            np.random.seed(seed)
            torch.random.default_generator.manual_seed(seed)
            if device.type == "cuda":
                with torch.cuda.device(device):
                    torch.cuda.manual_seed(seed)
            model.train()
            for module in model.modules():
                if isinstance(module, (torch.nn.modules.batchnorm._BatchNorm,
                                       torch.nn.modules.dropout._DropoutNd)):
                    module.eval()
            with torch.no_grad(), torch.autocast(device_type=device.type, enabled=False):
                yield
    finally:
        changed = False
        with torch.no_grad():
            for name, value in model.named_buffers():
                if not torch.equal(value, buffers[name]):
                    changed = True
                    value.copy_(buffers[name])
        for module, training in modes.items():
            module.training = training
        random.setstate(python_state)
        np.random.set_state(numpy_state)
        if changed:
            raise RuntimeError("validation loss 측정 중 버퍼가 바뀌어 복구했습니다. 모델 정규화 설정을 확인하세요.")


def validation_loss(model, dataset, config):
    """가중치 갱신 없이 validation 전체의 FP32 loss를 측정한다.

    Args:
        model (Module): 현재 epoch의 학습 모델.
        dataset (Dataset): 고정 validation 데이터.
        config (dict): 측정 batch와 고정 난수 설정.

    Returns:
        dict: 이미지 평균 전체 loss와 네 세부 loss.
    """
    device = next(model.parameters()).device
    loader = DataLoader(dataset, batch_size=config["val_loss_batch_size"], shuffle=False,
                        num_workers=config["num_workers"], collate_fn=collate_detection,
                        generator=torch.Generator().manual_seed(config["loss_measurement_seed"]),
                        pin_memory=device.type == "cuda")
    totals, seen = {}, 0
    expected = {"loss_classifier", "loss_box_reg", "loss_objectness", "loss_rpn_box_reg"}
    with _validation_loss_context(model, config["loss_measurement_seed"]):
        for images, targets in loader:
            values = model([image.to(device) for image in images],
                           [{key: value.to(device) for key, value in target.items()} for target in targets])
            if set(values) != expected:
                raise RuntimeError("검출 loss 네 종류가 필요합니다. Faster R-CNN 모델 구성을 확인하세요.")
            for key, value in values.items():
                if value.numel() != 1 or not torch.isfinite(value):
                    raise FloatingPointError("validation loss가 유한하지 않습니다. 라벨·모델을 확인하세요.")
                totals[key] = totals.get(key, 0.0) + float(value.detach()) * len(images)
            seen += len(images)
    if seen != len(dataset) or seen == 0:
        raise RuntimeError("validation 전체를 측정하지 못했습니다. 고정 분할을 확인하세요.")
    components = {f"val_{key}": value / seen for key, value in totals.items()}
    return {"val_loss": sum(components.values()), **components}



def _atomic_save(path, value):
    """완성된 체크포인트만 교체한다."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".part")
    torch.save(value, temporary)
    os.replace(temporary, path)


def _rng_state(generator):
    """실제로 사용하는 단일 GPU의 난수 상태만 보관한다."""
    return {"python": random.getstate(), "numpy": np.random.get_state(),
            "torch": torch.get_rng_state(), "loader": generator.get_state(),
            "cuda": torch.cuda.get_rng_state(0) if torch.cuda.is_available() else None}


def _restore_rng(state, generator):
    """장착된 전체 GPU 개수가 달라도 사용 GPU 0의 상태를 복원한다."""
    random.setstate(state["python"]); np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"]); generator.set_state(state["loader"])
    if state["cuda"] is not None:
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA 학습 체크포인트입니다. GPU 환경에서 재개하세요.")
        torch.cuda.set_rng_state(state["cuda"], 0)


def read_checkpoint(run_dir, signature):
    """현재 파일이 불완전하면 이전 정상 epoch와 짝이 맞는 best를 찾는다."""
    run_dir = Path(run_dir)
    failures = []
    for name in ("last.pt", "previous_last.pt"):
        path = run_dir / "weights" / name
        if not path.is_file():
            continue
        try:
            state = torch.load(path, map_location="cpu", weights_only=False)
            if state.get("signature") != signature:
                raise ValueError("실험 식별자가 다릅니다")
            required = {"model", "optimizer", "scheduler", "scaler", "rng", "epoch", "history", "best_epoch"}
            if not required.issubset(state) or not state["history"] or state["history"][-1]["epoch"] != state["epoch"]:
                raise ValueError("재개 상태·학습 이력이 불완전합니다")
            if not all(torch.isfinite(t).all() for t in state["model"].values() if t.is_floating_point()):
                raise ValueError("모델에 유한하지 않은 값이 있습니다")
            for best_name in ("best.pt", "previous_best.pt"):
                best_path = run_dir / "weights" / best_name
                if not best_path.is_file():
                    continue
                try:
                    best = torch.load(best_path, map_location="cpu", weights_only=False)
                except Exception:
                    continue
                if best.get("signature") == signature and best["epoch"] == state["best_epoch"]:
                    return state, path, best_path
            raise ValueError("현재 epoch와 일치하는 best 체크포인트가 없습니다")
        except Exception as error:
            failures.append(f"{name}: {type(error).__name__}")
    raise RuntimeError("사용할 정상 체크포인트가 없습니다. 공유 저장을 확인하세요. " + "; ".join(failures))


_register_notebook_block("model")

In [ ]:
def experiment_identity(bundle, plan, baseline):
    """경로 대신 실제 데이터·목록·가중치·실행 코드·lock·패키지를 재개 서명에 넣는다."""
    recipe = {"spec": plan["spec"], "config": plan["config"], "sampling": plan["sampling"],
              "class_mapping": {str(k): v for k, v in bundle["category_to_label"].items()},
              "data_identity": bundle["identity"], "class_frequency": plan["frequency"].to_dict("records"),
              "baseline_signature": baseline["manifest"]["signature"], "baseline_best_sha256": baseline["best_sha256"],
              "code": code_identity(), "uv_lock_sha256": file_sha256(REPO_ROOT / "uv.lock"),
              "packages": {n: importlib.metadata.version(n) for n in CORE_PACKAGES},
              "python": list(sys.version_info[:2]), "initialization": "independent_COCO_V1",
              "evaluation_weighted_loss": False}
    return stable_hash(recipe), recipe


def validate_saved_progress(state, plan, manifest):
    """재개 epoch·실제 갱신·노출 이력이 같은 예산의 정상 완료 prefix인지 확인한다."""
    epoch = state["epoch"]
    if type(epoch) is not int or not 1 <= epoch <= plan["config"]["max_epochs"]:
        raise ValueError("재개 epoch가 학습 예산 밖입니다. 정상 체크포인트를 확인하세요.")
    history = state["history"]
    if [r["epoch"] for r in history] != list(range(1, epoch + 1)) or state["run_id"] != manifest["run_id"]:
        raise ValueError("재개 이력·run ID가 다릅니다. 같은 run의 last를 확인하세요.")
    if any(r["optimizer_steps"] <= 0 or r["amp_skipped_steps"] < 0 or
           r["optimizer_steps"] + r["amp_skipped_steps"] != plan["sampling"]["steps_per_epoch"] for r in history):
        raise ValueError("저장된 실제 갱신·AMP skip 수가 예산과 다릅니다. 정상 last를 확인하세요.")
    winner = max(history, key=lambda r: r["val"])
    if state["best_epoch"] != winner["epoch"] or state["best"] != winner["val"]:
        raise ValueError("저장된 best 선택 기록이 다릅니다. best와 last의 짝을 확인하세요.")
    exposure = state.get("exposures", [])
    expected = {(e, c): count for e in range(1, epoch + 1) for c, count in plan["counts"].items()}
    got = {(r["epoch"], r["category_id"]): r["forwarded_objects"] for r in exposure}
    if (len(got) != len(exposure) or got != expected or any(
        not 0 <= r["successful_update_objects"] <= r["forwarded_objects"] for r in exposure)):
        raise ValueError("저장된 클래스 노출 이력이 불완전합니다. 정상 last를 확인하세요.")


def log_previews(run, records, bundle, epoch, count):
    """기준 run처럼 val 미리보기 3장을 W&B에 기록한다."""
    import wandb
    dataset = bundle["datasets"]["val"]
    images = []
    for record in records[:count]:
        anns = dataset.annotations[record["image_id"]]
        picture = draw_boxes(record["image_path"], [a["bbox"] for a in anns],
                             [str(a["category_id"]) for a in anns], color="lime")
        pred = record["prediction"]
        keep = torch.where(pred["scores"] >= 0.30)[0][:20]
        picture = draw_boxes(picture, [xyxy_to_xywh(pred["boxes"][i].tolist()) for i in keep],
                             [f"{int(pred['labels'][i])}:{float(pred['scores'][i]):.2f}" for i in keep], color="red")
        picture.thumbnail((1000, 1000))
        images.append(wandb.Image(picture, caption=f"epoch {epoch}"))
    run.log({"epoch": epoch, "Media/validation": images})


def _apply_weights(bundle, plan):
    """현재 조건의 학습에만 공용 RoI 가중치를 활성화한다."""
    use_fastrcnn_class_weights(bundle["root"], plan["sampling"]["weight_power"], plan["files"])
    return check_condition(bundle, plan)


def _batch_exposures(batches, reverse):
    """forward에 실제 투입된 GT 객체 노출 수. 내부 RoI 표본 수와 구분한다."""
    return Counter(reverse[int(label)] for _, targets in batches for target in targets for label in target["labels"])


def run_condition(bundle, plan, baseline, resume=False):
    """새 독립 조건 학습 또는 같은 조건의 정상 epoch 체크포인트 재개."""
    config, spec = plan["config"], plan["spec"]
    validate_config(config)
    validate_spec(spec)
    if PLATFORM != "colab" or not torch.cuda.is_available():
        raise RuntimeError("본 학습은 별도 CUDA Colab에서 실행하세요. 로컬에서는 학습하지 않습니다.")
    signature, recipe = experiment_identity(bundle, plan, baseline)
    run_dir = RUN_ROOT / spec["run_name"]
    manifest_path = run_dir / "config.json"
    state = source_state = source_best = None
    if run_dir.exists():
        if not manifest_path.is_file():
            raise FileExistsError("기존 폴더에 config.json이 없습니다. 지우거나 덮어쓰지 말고 새 RUN_SUFFIX를 지정하세요.")
        manifest = json.loads(manifest_path.read_text())
        if manifest.get("signature") != signature:
            raise ValueError("기존 run의 설정·코드·데이터·lock이 다릅니다. 동일 버전으로 재개하거나 새 RUN_SUFFIX를 쓰세요.")
        if (run_dir / "result.json").is_file():
            result = json.loads((run_dir / "result.json").read_text())
            best = torch.load(run_dir / "weights/best.pt", map_location="cpu", weights_only=False)
            if result["signature"] != signature or best["signature"] != signature or best["epoch"] != result["best_epoch"]:
                raise ValueError("완료 결과와 best 연결이 다릅니다. 저장 파일을 확인하세요.")
            print("완료 run 보존:", run_dir.name)
            return run_dir
        if not resume:
            raise FileExistsError("미완료 run이 있습니다. 9번 재개 셀에서 조건을 명시하세요.")
        if any((run_dir / "weights" / n).is_file() for n in ("last.pt", "previous_last.pt")):
            state, source_state, source_best = read_checkpoint(run_dir, signature)
            validate_saved_progress(state, plan, manifest)
        elif (run_dir / "metrics.csv").exists() or any((run_dir / "weights").glob("*.pt")):
            raise RuntimeError("학습 기록은 있지만 정상 last가 없습니다. 결과 폴더를 확인하세요.")
    elif resume:
        raise FileNotFoundError("재개할 run이 없습니다. 조건과 RUN_SUFFIX를 확인하세요.")
    else:
        run_dir.mkdir(parents=True, exist_ok=False)
        (run_dir / "weights").mkdir()
        manifest = {"signature": signature, "recipe": recipe, "spec": spec, "config": config,
                    "sampling": plan["sampling"], "run_id": uuid.uuid4().hex[:12],
                    "source": "new_COCO_initialization", "run_name": spec["run_name"]}
        write_json(manifest_path, manifest)
        write_json(run_dir / "baseline_snapshot.json", baseline_snapshot(baseline))
        write_json(run_dir / "data_fingerprint.json", bundle["fingerprint"])
        write_frame(run_dir / "class_map.csv", bundle["class_map"])
        write_frame(run_dir / "class_frequency.csv", plan["frequency"])
        write_json(run_dir / "sampling.json", plan["sampling"])
        write_json(run_dir / "train_files.json", plan["files"])
        for name in ("dataset_info.json", "classes.csv"):
            shutil.copy2(bundle["root"] / name, run_dir / name)
        shutil.copy2(bundle["root"] / "eval/val_gt.json", run_dir / "val_gt.json")
    history_path = run_dir / "environment_history.json"
    environments = json.loads(history_path.read_text()) if history_path.exists() else []
    environments.append({**runtime_info(), "resume": resume,
                         "checkpoint": source_state.name if source_state else None,
                         "baseline_package_differences": baseline["package_differences"]})
    write_json(history_path, environments)
    device = torch.device("cuda:0")
    seed_everything(0, config["deterministic"])
    generator = torch.Generator().manual_seed(0)
    loader = _loader(training_subset(bundle, plan), config, True, generator)
    steps = math.ceil(len(loader) / config["accumulation_steps"])
    if steps != plan["sampling"]["steps_per_epoch"]:
        raise ValueError("실제 DataLoader step 수가 계획과 다릅니다. 배치·누적·drop_last를 확인하세요.")
    model = build_model(len(bundle["class_map"]), spec, config, load_coco=state is None).to(device)
    optimizer = make_optimizer(model, spec, config)
    scheduler = make_scheduler(optimizer, steps, config)
    scaler = _make_scaler(config, device)
    history, exposures, best_val, best_epoch, elapsed_before = [], [], -1.0, 0, 0.0
    if state:
        model.load_state_dict(state["model"])
        scheduler.load_state_dict(state["scheduler"])
        optimizer.load_state_dict(state["optimizer"])
        scaler.load_state_dict(state["scaler"])
        history, exposures = state["history"], state["exposures"]
        best_val, best_epoch, elapsed_before = state["best"], state["best_epoch"], state["training_seconds"]
        if source_state.name != "last.pt":
            _atomic_save(run_dir / "weights/last.pt", state)
        if source_best.name != "best.pt":
            restored_best = torch.load(source_best, map_location="cpu", weights_only=False)
            _atomic_save(run_dir / "weights/best.pt", restored_best)
            del restored_best
    import wandb
    run = None
    started = time.perf_counter()
    try:
        run = wandb.init(entity=config["wandb_entity"], project=config["wandb_project"], id=manifest["run_id"],
            name=spec["run_name"], resume=("must" if state else "allow") if resume else "never",
            group="exp3_v3_frcnn_e10_budget", tags=["exp3", "v3", "frcnn", spec["condition"], "seed0"],
            config=recipe, dir=str(run_dir))
        if run is None or run.entity != config["wandb_entity"] or run.project != config["wandb_project"]:
            raise RuntimeError("팀 W&B 기록이 시작되지 않았습니다. 로그인과 팀 권한을 확인하세요.")
        run.define_metric("epoch")
        run.define_metric("*", step_metric="epoch")
        if state:
            _restore_rng(state["rng"], generator)
            start_epoch = state["epoch"] + 1
            del state
        else:
            seed_everything(0, config["deterministic"])
            start_epoch = 1
        for epoch in range(start_epoch, config["max_epochs"] + 1):
            tick = time.perf_counter()
            torch.cuda.reset_peak_memory_stats(device)
            applied = _apply_weights(bundle, plan)
            write_json(run_dir / "condition_applied.json", {**applied, "epoch": epoch, "steps_per_epoch": steps,
                       "loss_scope": "train_RoI_classifier_only", "validation_weighted": False})
            model.train()
            totals, seen, updates, skipped, micro_batches = {}, 0, 0, 0, 0
            forwarded, successful = Counter(), Counter()
            first_lr = optimizer.param_groups[0]["lr"]
            train_tick = time.perf_counter()
            for batches in _accumulation_groups(loader, config["accumulation_steps"]):
                batch_counts = _batch_exposures(batches, bundle["label_to_category"])
                group_images = sum(len(images) for images, _ in batches)
                losses, updated = _training_step(model, optimizer, scaler, batches, config)
                forwarded.update(batch_counts)
                if updated:
                    updates += 1
                    successful.update(batch_counts)
                    scheduler.step()
                else:
                    skipped += 1
                for key, value in losses.items():
                    totals[key] = totals.get(key, 0.0) + value * group_images
                seen += group_images
                micro_batches += len(batches)
            train_seconds = time.perf_counter() - train_tick
            optimizer.zero_grad(set_to_none=True)
            if seen != len(loader.dataset) or forwarded != plan["counts"] or updates + skipped != steps or updates == 0:
                raise RuntimeError("학습 목록·객체 노출·optimizer 갱신 집계가 다릅니다. 조건과 AMP 상태를 확인하세요.")
            # 전역 monkey patch는 validation/test에서 반드시 해제한다.
            use_fastrcnn_class_weights(bundle["root"], 0)
            measured = validation_loss(model, make_dataset(bundle, "val"), config)
            records, speed = collect_predictions(model, make_dataset(bundle, "val"), config,
                                                 bundle["label_to_category"], with_targets=False)
            scores = evaluate_val(predictions_for_team(records), bundle["gt"])
            if any(scores[k] is None or not math.isfinite(scores[k]) for k in ("val", "kaggle", "test_only")):
                raise RuntimeError("공식 점수 세 개가 유효하지 않습니다. 고정 val과 매핑을 확인하세요.")
            improved = scores["val"] > best_val
            if improved:
                best_val, best_epoch = scores["val"], epoch
            row = {"epoch": epoch, "train_loss": sum(totals.values()) / seen, **measured,
                   **{f"train_{k}": v / seen for k, v in totals.items()},
                   **{k: scores[k] for k in ("val", "kaggle", "test_only")},
                   "lr": first_lr, "last_lr": optimizer.param_groups[0]["lr"],
                   "train_seconds": train_seconds, "epoch_seconds": time.perf_counter() - tick,
                   "inference_ms": speed["model_ms_per_image"], "peak_gpu_gb": torch.cuda.max_memory_allocated(device) / 1024 ** 3,
                   "optimizer_steps": updates, "amp_skipped_steps": skipped, "micro_batches": micro_batches,
                   "train_images": seen, "last_group_images": group_images, "amp_scale": scaler.get_scale(),
                   "scheduler_steps": scheduler.last_epoch, **_batch_policy(config)}
            row.update({f"ap_K_{c:06d}": scores["per_class"].get(c) for c in MONITOR_CLASSES})
            history.append(row)
            exposures.extend({"epoch": epoch, "category_id": c, "expected_fixed_list": plan["counts"][c],
                              "forwarded_objects": forwarded[c], "successful_update_objects": successful[c]}
                             for c in sorted(plan["counts"]))
            weights_dir = run_dir / "weights"
            if improved:
                if (weights_dir / "best.pt").is_file():
                    shutil.copy2(weights_dir / "best.pt", weights_dir / "previous_best.pt")
                _atomic_save(weights_dir / "best.pt", {"signature": signature, "epoch": epoch,
                    "model": model.state_dict(), "scores": scores, "config": config, "spec": spec})
            checkpoint = {"signature": signature, "epoch": epoch, "model": model.state_dict(),
                "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
                "rng": _rng_state(generator), "best": best_val, "best_epoch": best_epoch, "history": history,
                "exposures": exposures, "config": config, "spec": spec, "run_id": manifest["run_id"],
                "training_seconds": elapsed_before + time.perf_counter() - started}
            if (weights_dir / "last.pt").is_file():
                shutil.copy2(weights_dir / "last.pt", weights_dir / "previous_last.pt")
            _atomic_save(weights_dir / "last.pt", checkpoint)
            write_frame(run_dir / "metrics.csv", pd.DataFrame(history))
            write_frame(run_dir / "class_exposure.csv", pd.DataFrame(exposures))
            run.log(row)
            if config["preview_count"]:
                try:
                    log_previews(run, records, bundle, epoch, config["preview_count"])
                except Exception as error:
                    print("미리보기 기록 실패:", type(error).__name__, "숫자·체크포인트는 보존했습니다.")
            _restore_rng(checkpoint["rng"], generator)
            print(f"{spec['condition']} {epoch}/{spec['epochs']} | loss {row['train_loss']:.4f} | "
                  f"val {scores['val']:.5f}, kaggle_val {scores['kaggle']:.5f}, "
                  f"test_only_val {scores['test_only']:.5f} | updates {updates}, AMP skip {skipped}", flush=True)
            del checkpoint, records
        if not history:
            raise RuntimeError("완료한 epoch가 없습니다. 저장된 last와 설정을 확인하세요.")
        result = {**spec, "signature": signature, "best_epoch": best_epoch, "last_epoch": history[-1]["epoch"],
                  "best_val": best_val, "state": "training_complete", "run_id": manifest["run_id"],
                  "training_seconds": elapsed_before + time.perf_counter() - started,
                  "optimizer_updates_planned": plan["sampling"]["total_optimizer_updates_planned"],
                  "optimizer_updates_actual": sum(r["optimizer_steps"] for r in history),
                  "amp_skipped_steps": sum(r["amp_skipped_steps"] for r in history),
                  "environment_confounded": baseline["environment_confounded"]}
        write_json(run_dir / "result.json", result)
        run.summary.update({"exp3/val": best_val, "best_epoch": best_epoch, "training_state": "complete",
                            "environment_confounded": baseline["environment_confounded"]})
        return run_dir
    finally:
        use_fastrcnn_class_weights(bundle["root"], 0)
        if run is not None:
            run.finish()
        del model, optimizer, scheduler, scaler
        gc.collect()
        torch.cuda.empty_cache()


_register_notebook_block("training")

In [ ]:
def prediction_cache(run_dir, bundle, plan, baseline, split="val"):
    """완료 run의 best에서 한 번 예측하고, 검증된 같은 캐시만 재사용한다."""
    run_dir = Path(run_dir)
    if split not in ("val", "test") or not (run_dir / "result.json").is_file():
        raise RuntimeError("완료 run의 val/test만 분석합니다. 학습 완료 상태를 확인하세요.")
    signature, _ = experiment_identity(bundle, plan, baseline)
    manifest = json.loads((run_dir / "config.json").read_text())
    result = json.loads((run_dir / "result.json").read_text())
    if manifest["signature"] != signature or result["signature"] != signature:
        raise ValueError("현재 실행 코드·환경·데이터가 학습과 다릅니다. 동일 버전으로 분석하세요.")
    dataset = make_dataset(bundle, "val") if split == "val" else TestImages(REPO_ROOT / "data/raw/test_images")
    if len(dataset) != (1000 if split == "val" else 842):
        raise ValueError("val/test 이미지 수가 다릅니다. 고정 데이터 경로를 확인하세요.")
    images = {r.file_name: image_digest(dataset.image_dir / r.file_name) for r in dataset.frame.itertuples()}
    best_path = run_dir / "weights/best.pt"
    identity = stable_hash({"signature": signature, "checkpoint": file_sha256(best_path),
                            "images": images, "split": split})
    cache, metadata, manifest_file = (run_dir / f"{split}_{name}" for name in
                                     ("predictions.json", "prediction_identity.json", "manifest.csv"))
    if any(p.exists() for p in (cache, metadata, manifest_file)):
        if not all(p.is_file() for p in (cache, metadata, manifest_file)):
            raise RuntimeError("예측 캐시 세 파일이 불완전합니다. 자동 덮어쓰기 없이 저장 상태를 확인하세요.")
        saved = json.loads(metadata.read_text())
        if (saved.get("identity") != identity or saved.get("checkpoint_epoch") != result["best_epoch"] or
            saved.get("predictions_sha256") != file_sha256(cache) or
            saved.get("manifest_sha256") != file_sha256(manifest_file)):
            raise ValueError("기존 예측 캐시가 현재 best·데이터와 다릅니다. 캐시를 덮어쓰지 말고 버전을 확인하세요.")
        return json.loads(cache.read_text())
    config, spec = plan["config"], plan["spec"]
    use_fastrcnn_class_weights(bundle["root"], 0)
    model = build_model(len(bundle["class_map"]), spec, config, load_coco=False).to("cuda:0")
    try:
        checkpoint = torch.load(best_path, map_location="cpu", weights_only=False)
        if checkpoint["signature"] != signature or checkpoint["epoch"] != result["best_epoch"]:
            raise ValueError("best 체크포인트와 완료 결과가 다릅니다. 저장 상태를 확인하세요.")
        model.load_state_dict(checkpoint["model"])
        del checkpoint
        records, speed = collect_predictions(model, dataset, config, bundle["label_to_category"], with_targets=False)
        if len(records) != len(dataset):
            raise RuntimeError("이미지 예측이 누락됐습니다. DataLoader와 파일을 확인하세요.")
        predictions = predictions_for_team(records)
        write_json(cache, predictions)
        write_frame(manifest_file, dataset.frame)
        write_json(metadata, {"identity": identity, "checkpoint_epoch": result["best_epoch"], "images": len(dataset),
                              "predictions_sha256": file_sha256(cache), "manifest_sha256": file_sha256(manifest_file),
                              "precision": config["inference_precision"], "speed": speed})
        return predictions
    finally:
        use_fastrcnn_class_weights(bundle["root"], 0)
        del model
        gc.collect()
        torch.cuda.empty_cache()


def evaluate_best(bundle, plan, baseline):
    """전체 val·기존 47장·test-only 클래스 val 및 클래스별 AP를 저장한다."""
    run_dir = RUN_ROOT / plan["spec"]["run_name"]
    predictions = prediction_cache(run_dir, bundle, plan, baseline, "val")
    scores = evaluate_val(predictions, bundle["gt"])
    checkpoint = torch.load(run_dir / "weights/best.pt", map_location="cpu", weights_only=False)
    reference = checkpoint["scores"]
    del checkpoint
    for key in ("val", "kaggle", "test_only"):
        if not math.isclose(scores[key], reference[key], rel_tol=0, abs_tol=1e-6):
            raise ValueError(f"재추론한 {key}가 best 기록과 다릅니다. 환경·정밀도·데이터를 확인하세요.")
    write_json(run_dir / "scores.json", scores)
    rows = class_ap_rows(scores["per_class"], bundle["root"] / "classes.csv", "v3")
    write_frame(run_dir / "class_ap.csv", pd.DataFrame(rows))
    write_json(run_dir / "imbalance.json", imbalance_summary(scores["per_class"], bundle["root"] / "classes.csv", "v3"))
    write_json(run_dir / "analysis_complete.json", {"signature": checkpoint_signature(run_dir),
               "scores_sha256": file_sha256(run_dir / "scores.json"),
               "prediction_identity": json.loads((run_dir / "val_prediction_identity.json").read_text())["identity"]})
    return scores


def checkpoint_signature(run_dir):
    """실행 manifest의 서명을 읽는다."""
    return json.loads((Path(run_dir) / "config.json").read_text())["signature"]


def compare_runs(bundle, plans, baseline, public_scores=None):
    """고정된 원래 학습 빈도를 기준으로 baseline과 완료 조건을 비교한다."""
    public_scores = public_scores or {}
    baseline_predictions = json.loads((BASELINE_DIR / "val_predictions.json").read_text())
    baseline_scores = evaluate_val(baseline_predictions, bundle["gt"])
    for key in ("val", "kaggle", "test_only"):
        if not math.isclose(baseline_scores[key], baseline["scores"][key], rel_tol=0, abs_tol=1e-10):
            raise ValueError("baseline 저장 예측의 재채점이 원본 점수와 다릅니다. 평가 코드를 확인하세요.")
    class_table = bundle["root"] / "classes.csv"
    base_imbalance = imbalance_summary(baseline_scores["per_class"], class_table, "v3")
    rows = [{"condition": "baseline_exp2", **{k: baseline_scores[k] for k in ("val", "kaggle", "test_only")},
             "actual_kaggle_public": baseline["actual_public_score"], "delta_val": 0.0,
             "optimizer_updates_actual": baseline["actual_optimizer_updates"],
             "optimizer_updates_planned": 5610, "amp_skipped_steps": baseline["amp_skipped_steps"],
             "training_seconds": baseline["result"]["training_seconds"], **base_imbalance,
             "environment_confounded": baseline["environment_confounded"]}]
    base_rows = class_ap_rows(baseline_scores["per_class"], class_table, "v3")
    class_frame = pd.DataFrame(base_rows).rename(columns={"ap": "baseline_ap"})
    for condition, plan in plans.items():
        run_dir = RUN_ROOT / plan["spec"]["run_name"]
        if not (run_dir / "analysis_complete.json").is_file():
            print("미완료/미분석 조건:", condition)
            continue
        signature, _ = experiment_identity(bundle, plan, baseline)
        analysis = json.loads((run_dir / "analysis_complete.json").read_text())
        if analysis["signature"] != signature or analysis["scores_sha256"] != file_sha256(run_dir / "scores.json"):
            raise ValueError("분석 결과 서명·점수 해시가 다릅니다. 같은 버전으로 다시 확인하세요.")
        scores = json.loads((run_dir / "scores.json").read_text())
        result = json.loads((run_dir / "result.json").read_text())
        ap = {int(k): v for k, v in scores["per_class"].items()}
        class_frame[f"{condition}_ap"] = class_frame.category_id.map(ap)
        class_frame[f"{condition}_delta"] = class_frame[f"{condition}_ap"] - class_frame.baseline_ap
        imbalance = imbalance_summary(scores["per_class"], class_table, "v3")
        public = public_scores.get(condition)
        if public is not None and (not isinstance(public, (int, float)) or not math.isfinite(public) or not 0 <= public <= 1):
            raise ValueError("실제 제출 점수는 0~1의 숫자 또는 None으로 적으세요.")
        rows.append({"condition": condition, **{k: scores[k] for k in ("val", "kaggle", "test_only")},
            "delta_val": scores["val"] - baseline_scores["val"], "actual_kaggle_public": public,
            "delta_kaggle_public": None if public is None else public - baseline["actual_public_score"],
            "optimizer_updates_actual": result["optimizer_updates_actual"],
            "optimizer_updates_planned": result["optimizer_updates_planned"],
            "amp_skipped_steps": result["amp_skipped_steps"], "training_seconds": result["training_seconds"],
            "environment_confounded": baseline["environment_confounded"], **imbalance})
    out = RUN_ROOT / "comparisons" / (time.strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8])
    out.mkdir(parents=True, exist_ok=False)
    summary = pd.DataFrame(rows)
    monitored = class_frame[class_frame.category_id.isin(MONITOR_CLASSES)].copy()
    write_frame(out / "summary.csv", summary)
    write_frame(out / "class_ap_comparison.csv", class_frame)
    write_frame(out / "monitor_classes.csv", monitored)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    summary.plot.bar(x="condition", y="val", ax=axes[0], legend=False, title="Overall validation AP75:95")
    summary.plot.bar(x="condition", y=["rare_ap", "frequent_ap"], ax=axes[1], title="Original-train frequency cohorts")
    save_figure(out / "ap_comparison.png", fig)
    if len(monitored):
        ap_columns = [c for c in monitored.columns if c.endswith("_ap")]
        fig, ax = plt.subplots(figsize=(11, 4))
        monitored.set_index("k_code")[ap_columns].plot.bar(ax=ax, title="Four monitoring classes")
        save_figure(out / "monitor_classes.png", fig)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for name, history in [("baseline", baseline["metrics"])] + [
        (c, pd.read_csv(RUN_ROOT / p["spec"]["run_name"] / "metrics.csv")) for c, p in plans.items()
        if (RUN_ROOT / p["spec"]["run_name"] / "metrics.csv").is_file()]:
        for ax, metric in zip(axes, ("train_loss", "val", "last_lr")):
            ax.plot(history.epoch, history[metric], label=name)
            ax.set_title(metric)
            ax.set_xlabel("epoch (length differs for RFS)")
    for ax in axes:
        ax.legend()
    save_figure(out / "learning_curves.png", fig)
    write_json(out / "comparison_complete.json", {"baseline": baseline_snapshot(baseline),
        "conditions": summary.condition.tolist(), "notes": [
            "kaggle 컬럼은 validation 47장의 점수이며 actual_kaggle_public과 다름",
            "test_only는 해당 클래스 validation 점수이며 숨겨진 test 정답 점수가 아님",
            "baseline과 현재 uv.lock 환경 차이가 있어 기법만의 인과 효과로 단정할 수 없음",
            "단일 seed의 작은 val 변화는 통계적 유의성이나 test 개선을 보장하지 않음",
            "예정 update가 비슷해도 실제 연산량·소요 시간이 동일하지 않음"]})
    display(summary)
    display(monitored)
    print("새 비교 결과:", out)
    return out


_register_notebook_block("analysis")

## 4. Drive와 W&B 연결
`.env`를 읽지 않습니다. 경로는 1번에서 지정하고, W&B 키는 사용자가 로그인 UI에 직접 입력합니다. API 키는 설정·출력·저장 파일에 넣지 않습니다.

In [ ]:
MOUNT_DRIVE = True
LOGIN_WANDB = True
if MOUNT_DRIVE:
    if PLATFORM != "colab":
        raise RuntimeError("Drive 마운트는 Colab에서만 실행하세요.")
    from google.colab import drive
    drive.mount("/content/drive")
if LOGIN_WANDB:
    import wandb
    os.environ["WANDB_ENTITY"] = "codeit-prj1-team2"
    wandb.login()
if not DRIVE_BASE.is_dir():
    print("공유 폴더 경로가 아직 없습니다. 마운트와 1번 DRIVE_BASE를 먼저 확인하세요.")

## 5. 데이터 확보·고정 목록 확인·v3 생성
2차와 같은 팀 ZIP과 `make_dataset_lists.py --check`, `build_dataset.py --variant v3 --format coco`를 사용합니다. 기존 파일은 크기·CRC가 같을 때만 재사용하며 다르면 중단합니다. 고정 CSV와 원본 정제 목록은 ZIP에서 덮어쓰지 않습니다. test ZIP은 제출 CSV가 필요할 때 별도로 준비합니다.

In [ ]:
import stat
import zlib


def extract_team_zip(archive_name):
    """허용된 팀 데이터만 전개하고, 기존 파일이 다르면 덮어쓰지 않는다."""
    if PLATFORM != "colab":
        raise RuntimeError("로컬 데이터는 변경하지 않습니다. 새 Colab에서 실행하세요.")
    allowed = {"kaggle.zip": ("data/raw/train_images/", "data/processed/train_annotations/"),
               "aihub_annotations.zip": ("data/processed/ai_hub/train_annotations/",),
               "aihub_images.zip": ("data/processed/ai_hub/train_images/",),
               "test_images.zip": ("data/raw/test_images/",)}
    if archive_name not in allowed:
        raise ValueError("허용하지 않은 ZIP입니다. 팀 공유 ZIP 4개만 사용하세요.")
    archive = ZIP_ROOT / archive_name
    if not archive.is_file():
        raise FileNotFoundError(f"ZIP이 없습니다: {archive}. Drive 바로가기와 ZIP_ROOT를 확인하세요.")
    with zipfile.ZipFile(archive) as source:
        for member in source.infolist():
            parts = Path(member.filename).parts
            if "__MACOSX" in parts or ".DS_Store" in parts or any(p.startswith("._") for p in parts) or member.is_dir():
                continue
            if not member.filename.startswith(allowed[archive_name]):
                if member.filename in ("data/processed/split.csv", "data/split.csv"):
                    continue  # 고정 분할은 저장소 CSV를 사용하며 ZIP의 옛 split은 적용하지 않는다.
                raise ValueError(f"예상 범위 밖 ZIP 항목: {member.filename}. 팀 ZIP 구성을 확인하세요.")
            destination = (REPO_ROOT / member.filename).resolve()
            if not destination.is_relative_to((REPO_ROOT / "data").resolve()) or stat.S_ISLNK(member.external_attr >> 16):
                raise ValueError("ZIP 경로·심볼릭 링크가 안전하지 않습니다. 팀 ZIP을 확인하세요.")
            if destination.exists():
                crc = 0
                with destination.open("rb") as stream:
                    for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b""):
                        crc = zlib.crc32(chunk, crc)
                if destination.stat().st_size != member.file_size or crc != member.CRC:
                    raise ValueError(f"기존 파일과 ZIP이 다릅니다: {member.filename}. 자동 덮어쓰기 없이 확인하세요.")
                continue
            destination.parent.mkdir(parents=True, exist_ok=True)
            temporary = destination.with_name(destination.name + ".part")
            with source.open(member) as input_stream, temporary.open("wb") as output_stream:
                shutil.copyfileobj(input_stream, output_stream)
            os.replace(temporary, destination)
    print("ZIP 확인·전개 완료:", archive_name)


PREPARE_DATA = False
BUILD_V3 = False
PREPARE_TEST_DATA = True
if PREPARE_DATA:
    for archive_name in ("kaggle.zip", "aihub_annotations.zip", "aihub_images.zip"):
        extract_team_zip(archive_name)
    environment = {**os.environ, "PYTHONPATH": str(REPO_ROOT / "src"), "PYTHONDONTWRITEBYTECODE": "1"}
    checked = subprocess.run([sys.executable, str(REPO_ROOT / "scripts/make_dataset_lists.py"), "--check"],
                             cwd=REPO_ROOT, env=environment, capture_output=True, text=True)
    print(checked.stdout)
    checked.check_returncode()
    if "images.csv: 같음" not in checked.stdout or "classes.csv: 같음" not in checked.stdout:
        raise RuntimeError("고정 목록이 같지 않습니다. CSV를 생성·수정하지 말고 데이터를 확인하세요.")
if BUILD_V3:
    if PLATFORM != "colab":
        raise RuntimeError("학습 폴더 생성은 새 Colab에서 실행하세요.")
    out = DATASET_ROOT / "v3"
    if out.exists() and any(out.iterdir()):
        print("기존 v3 폴더를 보존하고 다음 셀에서 baseline 지문과 대조합니다:", out)
    else:
        subprocess.run([sys.executable, str(REPO_ROOT / "scripts/build_dataset.py"), "--variant", "v3",
                        "--format", "coco", "--out", str(out)], cwd=REPO_ROOT,
                       env={**os.environ, "PYTHONPATH": str(REPO_ROOT / "src")}, check=True)
if PREPARE_TEST_DATA:
    extract_team_zip("test_images.zip")

## 6. baseline과 세 조건 계획 확인
원본 baseline의 SHA-256, best·예측·정답 연결, 현재 v3 이미지 내용까지 대조합니다. RFS 목록은 공용 기본식과 seed 0으로 한 번 정해 재사용합니다. 기대 노출은 확률적 반복계수 기준이고, 고정 목록의 노출과 실제 forward·성공 갱신 노출은 따로 저장합니다.

In [ ]:
INSPECT_AND_PLAN = True
if INSPECT_AND_PLAN:
    BASELINE = load_baseline()
    BUNDLE = prepare_bundle("v3")
    PLANS = make_plans(BUNDLE, BASELINE)
    display(pd.DataFrame([{"condition": c, **p["sampling"]} for c, p in PLANS.items()])[
        ["condition", "n_list", "epochs", "steps_per_epoch", "total_optimizer_updates_planned",
         "updates_difference", "updates_ratio", "warmup_epochs", "warmup_steps", "baseline_warmup_steps"]])
    print("baseline 실제 optimizer 갱신:", BASELINE["actual_optimizer_updates"], "AMP skip:", BASELINE["amp_skipped_steps"])
    print("baseline과 패키지 차이:", BASELINE["package_differences"])
    for c, p in PLANS.items():
        print(c, "배경 가중치", p["sampling"]["background_weight"])
        display(p["frequency"][p["frequency"].monitor])

In [ ]:
BATCH_REPORTS = {}


def require_plan():
    """준비 셀 누락을 이름 오류 대신 실행 순서로 안내한다."""
    if any(name not in globals() for name in ("BASELINE", "BUNDLE", "PLANS")):
        raise RuntimeError("6번 INSPECT_AND_PLAN=True로 baseline·v3·세 조건을 먼저 확인하세요.")


def batch_probe(bundle, plan, baseline):
    """본 실험과 분리한 배치 16·AMP 갱신 검사. 검사 모델은 본 실험에 재사용하지 않는다."""
    if PLATFORM != "colab" or not torch.cuda.is_available():
        raise RuntimeError("배치 검사는 별도 CUDA Colab에서 사용자가 실행하세요.")
    config, spec = plan["config"], plan["spec"]
    signature, _ = experiment_identity(bundle, plan, baseline)
    seed_everything(0, config["deterministic"])
    device = torch.device("cuda:0")
    generator = torch.Generator().manual_seed(0)
    dataset = training_subset(bundle, plan)
    indexes = torch.randperm(len(dataset), generator=generator)[:16].tolist()
    loader = _loader(Subset(dataset, indexes), config, True, generator)
    batches = [next(iter(loader))]
    model = build_model(len(bundle["class_map"]), spec, config).to(device)
    optimizer = make_optimizer(model, spec, config)
    scaler = _make_scaler(config, device)
    before = model.roi_heads.box_predictor.cls_score.weight.detach().clone()
    try:
        applied = _apply_weights(bundle, plan)
        torch.cuda.reset_peak_memory_stats(device)
        updated = False
        for attempt in range(8):
            model.train()
            losses, updated = _training_step(model, optimizer, scaler, batches, config)
            if updated:
                break
        if not updated or torch.equal(before, model.roi_heads.box_predictor.cls_score.weight.detach()):
            raise RuntimeError("AMP 갱신이 성공하지 않았습니다. 수치·GPU 상태를 확인하세요.")
        use_fastrcnn_class_weights(bundle["root"], 0)
        measured = validation_loss(model, Subset(make_dataset(bundle, "val"), list(range(8))), config)
        report = {"signature": signature, "applied": applied, "batch_size": 16,
                  "attempts": attempt + 1, "losses": losses, "validation_loss": measured,
                  "peak_gpu_gb": torch.cuda.max_memory_allocated(device) / 1024 ** 3,
                  "probe_weights_reused": False, "status": "passed"}
        out = RUN_ROOT / "checks" / (spec["condition"] + "_" + uuid.uuid4().hex[:8])
        out.mkdir(parents=True, exist_ok=False)
        write_json(out / "report.json", report)
        BATCH_REPORTS[signature] = report
        print("별도 배치 검사 통과:", spec["condition"], "|", out)
        return report
    finally:
        use_fastrcnn_class_weights(bundle["root"], 0)
        del model, optimizer, scaler, before, batches
        gc.collect()
        torch.cuda.empty_cache()


_register_notebook_block("checks")

## 7. 선택: 조건 적용과 실제 배치 16 검사
전체 데이터 미리보기와 별도로 조건별 가중치·반복 길이를 확인합니다. `CHECK_BATCH=True`는 조건마다 독립 임시 모델에서 AMP 갱신과 FP32 검증 loss를 검사합니다. 결과는 `cj_runs/exp3/checks/`에 저장하고 검사 가중치는 본 실험에 사용하지 않습니다. OOM 시 자동으로 배치를 낮추지 않습니다.

In [ ]:
CHECK_CONDITIONS = True
CHECK_BATCH = True
if CHECK_CONDITIONS or CHECK_BATCH:
    require_plan()
    try:
        for condition in CONDITIONS:
            print(_apply_weights(BUNDLE, PLANS[condition]))
    finally:
        use_fastrcnn_class_weights(BUNDLE["root"], 0)
if CHECK_BATCH:
    for condition in CONDITIONS:
        batch_probe(BUNDLE, PLANS[condition], BASELINE)

## 8. 본 학습
배치 검사를 통과한 조건만 실행합니다. 각 조건은 동일한 seed·COCO_V1 초기화 정책에서 새로 시작합니다. 기존 미완료 run은 자동 재개하지 않으며, 완료 run은 보존합니다. `EVALUATE_AFTER_TRAINING=True`를 함께 선택하면 각 run 완료 직후 best를 재추론해 캐시·클래스 AP를 저장합니다. 그렇지 않으면 10번에서 실행합니다.

In [ ]:
START_TRAINING = True
EVALUATE_AFTER_TRAINING = True
SELECTED_CONDITIONS = ("rfs", "weight", "both")
if START_TRAINING:
    require_plan()
    if not SELECTED_CONDITIONS or set(SELECTED_CONDITIONS) - set(CONDITIONS):
        raise ValueError("SELECTED_CONDITIONS에는 rfs·weight·both만 지정하세요.")
    for condition in SELECTED_CONDITIONS:
        plan = PLANS[condition]
        signature, _ = experiment_identity(BUNDLE, plan, BASELINE)
        if signature not in BATCH_REPORTS:
            raise RuntimeError(f"{condition} 배치 검사가 없습니다. 7번 CHECK_BATCH=True로 먼저 확인하세요.")
        run_condition(BUNDLE, plan, BASELINE, resume=False)
        if EVALUATE_AFTER_TRAINING:
            evaluate_best(BUNDLE, plan, BASELINE)

## 9. 같은 run 재개
새 Colab 세션에서는 동일한 브랜치·lock·노트북 함수·Drive·데이터 준비 후 6번까지 다시 실행합니다. 재개할 조건만 명시하세요. 완료 epoch의 model·optimizer·scheduler·AMP scaler·난수·DataLoader generator·객체 노출 이력을 복원합니다. 끊긴 epoch는 다시 실행합니다. 설정·epoch 예산을 바꾸는 재개는 허용하지 않습니다.

In [ ]:
RESUME_TRAINING = False
RESUME_CONDITIONS = ()  # 예: ("rfs",)
if RESUME_TRAINING:
    require_plan()
    if not RESUME_CONDITIONS or set(RESUME_CONDITIONS) - set(CONDITIONS):
        raise ValueError("RESUME_CONDITIONS에 재개할 조건을 명시하세요.")
    for condition in RESUME_CONDITIONS:
        run_condition(BUNDLE, PLANS[condition], BASELINE, resume=True)

## 10. best 평가·baseline 비교
이 셀은 GPU 재추론을 포함할 수 있습니다. 같은 best·이미지·서명의 검증된 캐시가 있으면 재사용합니다. baseline은 검증된 원본 예측을 같은 공용 평가 함수로 다시 채점하며 모델을 재학습하거나 재추론하지 않습니다. 희소/빈번 집단은 **원래 v3 학습 객체 수**로 고정합니다.

`kaggle`는 기존 validation 47장 점수, `test_only`는 해당 클래스의 validation 점수입니다. 실제 제출 점수는 아래 `ACTUAL_PUBLIC_SCORES`에 제출 후 직접 입력합니다. 비교 표·그림은 새 시각/식별자 폴더에 저장합니다.

In [ ]:
ANALYZE_BEST = False
ANALYZE_CONDITIONS = ("rfs", "weight", "both")
COMPARE_RESULTS = False
ACTUAL_PUBLIC_SCORES = {"rfs": None, "weight": None, "both": None}
if ANALYZE_BEST:
    require_plan()
    if not ANALYZE_CONDITIONS or set(ANALYZE_CONDITIONS) - set(CONDITIONS):
        raise ValueError("ANALYZE_CONDITIONS에 분석할 완료 조건만 적으세요.")
    for condition in ANALYZE_CONDITIONS:
        evaluate_best(BUNDLE, PLANS[condition], BASELINE)
if COMPARE_RESULTS:
    require_plan()
    comparison_dir = compare_runs(BUNDLE, PLANS, BASELINE, ACTUAL_PUBLIC_SCORES)

## 11. 선택: test 예측 캐시와 제출 CSV 생성
정답 없는 test 842장은 구성·매핑을 유지합니다. score 0.001부터 예측을 캐시하고, 제출 CSV에는 2차 참고 노트북과 같은 score 0.01 이상·대회 74클래스만 남깁니다. image_id는 숫자 파일명, category_id는 원래 K-코드, bbox는 원본 픽셀 xywh입니다. **CSV 생성과 Kaggle 제출은 별개이며 이 노트북은 제출하지 않습니다.** 제출 한도와 남은 횟수는 사용자가 실제 제출할 때 확인합니다.

In [ ]:
CREATE_SUBMISSION = True
SUBMISSION_CONDITION = ""  # val·실제 결과를 보고 rfs, weight, both 중 하나를 직접 선택
SUBMISSION_SCORE = 0.01
if CREATE_SUBMISSION:
    require_plan()
    if SUBMISSION_CONDITION not in CONDITIONS or SUBMISSION_SCORE != 0.01:
        raise ValueError("조건을 직접 선택하고 2차와 같은 제출 score 0.01을 사용하세요.")
    plan = PLANS[SUBMISSION_CONDITION]
    run_dir = RUN_ROOT / plan["spec"]["run_name"]
    predictions = prediction_cache(run_dir, BUNDLE, plan, BASELINE, "test")
    chosen = [p for p in predictions if p["score"] >= SUBMISSION_SCORE]
    out = run_dir / "submissions" / ("exp3_" + plan["spec"]["run_name"] + ".csv")
    competition = competition_from_classes(BUNDLE["root"] / "classes.csv")
    if out.exists():
        raise FileExistsError(f"제출 CSV가 이미 있습니다: {out}. 기존 파일을 확인하세요.")
    count = write_submission(chosen, out, competition)
    write_json(out.with_suffix(".manifest.json"), {"signature": checkpoint_signature(run_dir),
        "condition": SUBMISSION_CONDITION, "score": SUBMISSION_SCORE,
        "test_prediction_identity": json.loads((run_dir / "test_prediction_identity.json").read_text())["identity"],
        "csv_sha256": file_sha256(out), "rows": count, "submitted": False})
    print("CSV 생성 완료:", out, "|", count, "줄 | Kaggle 제출은 사용자가 직접 수행합니다.")

## 저장 결과와 해석
각 run에는 실제 config·환경 이력·데이터 지문·클래스 매핑·목록·가중치·기대 노출·epoch별 실제 forward/성공 갱신 노출·손실·공식 세 점수·LR·시간·best/last/이전 정상 체크포인트가 남습니다. 10번 이후에는 best 예측·클래스별 AP·불균형 지표와 비교 자료가 추가됩니다.

val이 거의 포화되어 작은 상승만으로 test 개선을 판단하지 않습니다. 단일 seed에서 통계적 유의성을 주장하지 않습니다. 효과가 없더라도 모든 불균형 완화 기법으로 일반화하지 않습니다. rfs/both는 예정 5,715회로 baseline 예정 5,610회보다 1.87% 많고, warmup은 1 step 많습니다. 실제 갱신 횟수는 AMP skip 때문에 달라지며, 비슷한 step 수가 동일한 연산량·시간을 뜻하지 않습니다.

공용 코드·기존 노트북·고정 CSV·원본 baseline에는 쓰지 않습니다. 새 학습은 별도 Colab에서 사용자가 실행합니다.